# 1. Classification Tree Code

In [1]:
from sklearn.preprocessing import OneHotEncoder
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, adjusted_rand_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from joblib import Parallel, delayed
from sklearn.preprocessing import LabelEncoder

class DecisionTree:
    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1, criterion='entropy'):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.criterion = criterion
        self.tree = None                                                          
        self.feature_importances = None                                          


    def entropy(self, y):
        counts = np.bincount(y)                                                  
        probabilities = counts / len(y)                                          
        return -np.sum([p * np.log2(p) for p in probabilities if p > 0])         


    def gini(self, y):
        counts = np.bincount(y)
        probabilities = counts / len(y)
        return 1 - np.sum(probabilities ** 2)


    def information_gain(self, y, left_indices, right_indices):
        if self.criterion == 'entropy':                                          
            impurity_func = self.entropy
        elif self.criterion == 'gini':
            impurity_func = self.gini
        else:
            raise ValueError(f"Unknown criterion: {self.criterion}")

        parent_impurity = impurity_func(y)                                       
        left_impurity = impurity_func(y[left_indices])
        right_impurity = impurity_func(y[right_indices])

        n, n_left, n_right = len(y), len(left_indices), len(right_indices)
        weighted_impurity = (n_left / n) * left_impurity + (n_right / n) * right_impurity
        inf_gain = parent_impurity - weighted_impurity
        
        return inf_gain                                                          
    
    
    def custom_1(self, y_oh, left_indices, right_indices):
        N = y_oh.sum()

        left = y_oh[left_indices]
        right = y_oh[right_indices]
        p_1 = left.sum() / N
        p_2 = right.sum() / N
        num_classes = y_oh.shape[1]                                               

        sum_total = 0
        epsilon = 1e-10
        
        for l in range(num_classes):
            p_1l = left[:, l].sum() / N
            p_2l = right[:, l].sum() / N
            p_l = p_1l + p_2l
            b = 1
            
            denominator_1 = max(p_1 * b**2, epsilon)
            denominator_2 = max(p_2 * b**2, epsilon)
            
            sum_total += ((p_1l - p_1 * p_l)**2) / denominator_1
            sum_total += ((p_2l - p_2 * p_l)**2) / denominator_2

        return N * sum_total
    
    
    def custom_2(self, y_oh, left_indices, right_indices):
        N = y_oh.sum()

        left = y_oh[left_indices]
        right = y_oh[right_indices]
        p_1 = left.sum() / N
        p_2 = right.sum() / N
        num_classes = y_oh.shape[1]                                             

        sum_total = 0
        epsilon = 1e-10 
        
        for l in range(num_classes):
            p_1l = left[:, l].sum() / N
            p_2l = right[:, l].sum() / N
            p_l = p_1l + p_2l
            
            b = np.sqrt(p_l)
            
            denominator_1 = max(p_1 * b**2, epsilon)
            denominator_2 = max(p_2 * b**2, epsilon)
            
            sum_total += ((p_1l - p_1 * p_l)**2) / denominator_1
            sum_total += ((p_2l - p_2 * p_l)**2) / denominator_2

        return N * sum_total
    

    def custom_3(self, y_oh, left_indices, right_indices):
        N = y_oh.sum()

        left = y_oh[left_indices]
        right = y_oh[right_indices]
        p_1 = left.sum() / N
        p_2 = right.sum() / N
        num_classes = y_oh.shape[1]                                              

        sum_total = 0
        epsilon = 1e-10 
        
        for l in range(num_classes):
            p_1l = left[:, l].sum() / N
            p_2l = right[:, l].sum() / N
            p_l = p_1l + p_2l
            
            b = np.sqrt(p_l*(1 - p_l))
            
            denominator_1 = max(p_1 * b**2, epsilon)
            denominator_2 = max(p_2 * b**2, epsilon)
            
            sum_total += ((p_1l - p_1 * p_l)**2) / denominator_1
            sum_total += ((p_2l - p_2 * p_l)**2) / denominator_2

        return N * sum_total
    

    def custom_4(self, y_oh, left_indices, right_indices):
        N = y_oh.sum()

        left = y_oh[left_indices]
        right = y_oh[right_indices]
        p_1 = left.sum() / N
        p_2 = right.sum() / N
        num_classes = y_oh.shape[1]                                               

        sum_total = 0
        epsilon = 1e-10
        
        for l in range(num_classes):
            p_1l = left[:, l].sum() / N
            p_2l = right[:, l].sum() / N
            p_l = p_1l + p_2l
            
            b = p_l
            
            denominator_1 = max(p_1 * b**2, epsilon)
            denominator_2 = max(p_2 * b**2, epsilon)
            
            sum_total += ((p_1l - p_1 * p_l)**2) / denominator_1
            sum_total += ((p_2l - p_2 * p_l)**2) / denominator_2


        return N * sum_total
    
    
    def custom_5(self, y_oh, left_indices, right_indices):
        N = y_oh.sum()

        left = y_oh[left_indices]
        right = y_oh[right_indices]
        p_1 = left.sum() / N
        p_2 = right.sum() / N
        num_classes = y_oh.shape[1]                                              

        sum_total = 0
        epsilon = 1e-10
        
        for l in range(num_classes):
            p_1l = left[:, l].sum() / N
            p_2l = right[:, l].sum() / N
            p_l = p_1l + p_2l
            
            b = p_l**2
            
            denominator_1 = max(p_1 * b**2, epsilon)
            denominator_2 = max(p_2 * b**2, epsilon)
            
            sum_total += ((p_1l - p_1 * p_l)**2) / denominator_1
            sum_total += ((p_2l - p_2 * p_l)**2) / denominator_2

        return N * sum_total
    

    def custom_6(self, y_oh, left_indices, right_indices):
        N = y_oh.sum()

        left = y_oh[left_indices]
        right = y_oh[right_indices]
        p_1 = left.sum() / N
        p_2 = right.sum() / N
        num_classes = y_oh.shape[1]                                              

        sum_total = 0
        epsilon = 1e-10
        
        for l in range(num_classes):
            p_1l = left[:, l].sum() / N
            p_2l = right[:, l].sum() / N
            p_l = p_1l + p_2l
            
            b = -np.log(max(p_l, epsilon))
            
            denominator_1 = max(p_1 * b**2, epsilon)
            denominator_2 = max(p_2 * b**2, epsilon)
            
            sum_total += ((p_1l - p_1 * p_l)**2) / denominator_1
            sum_total += ((p_2l - p_2 * p_l)**2) / denominator_2

        return N * sum_total
    

    def custom_7(self, y_oh, left_indices, right_indices):
        N = y_oh.sum()

        left = y_oh[left_indices]
        right = y_oh[right_indices]
        p_1 = left.sum() / N
        p_2 = right.sum() / N
        num_classes = y_oh.shape[1]                                               

        sum_total = 0
        epsilon = 1e-10
        
        for l in range(num_classes):
            p_1l = left[:, l].sum() / N
            p_2l = right[:, l].sum() / N
            p_l = p_1l + p_2l
            b = -(p_l**0.5) * np.log(max(p_l, epsilon))
            
            denominator_1 = max(p_1 * b**2, epsilon)
            denominator_2 = max(p_2 * b**2, epsilon)
            
            sum_total += ((p_1l - p_1 * p_l)**2) / denominator_1
            sum_total += ((p_2l - p_2 * p_l)**2) / denominator_2

        return N * sum_total
    
    
    def custom_8(self, y_oh, left_indices, right_indices):
        N = y_oh.sum()

        left = y_oh[left_indices]
        right = y_oh[right_indices]
        p_1 = left.sum() / N
        p_2 = right.sum() / N
        num_classes = y_oh.shape[1]                                               

        sum_total = 0
        epsilon = 1e-10
        
        for l in range(num_classes):
            p_1l = left[:, l].sum() / N
            p_2l = right[:, l].sum() / N
            p_l = p_1l + p_2l
            b = (-p_l)*np.log(max(p_l, epsilon))
            
            denominator_1 = max(p_1 * b**2, epsilon)
            denominator_2 = max(p_2 * b**2, epsilon)
            
            sum_total += ((p_1l - p_1 * p_l)**2) / denominator_1
            sum_total += ((p_2l - p_2 * p_l)**2) / denominator_2

        return N * sum_total    
    

    def most_common_label(self, y):
        return Counter(y).most_common(1)[0][0]


    def find_best_split(self, X, y, num_features, y_oh=None):
        best_gain = -float('inf')                                                  
        best_split = None                                                          

        for feature_index in range(num_features):                                  
            feature_values = np.sort(X[:, feature_index])
            thresholds = (feature_values[:-1] + feature_values[1:]) / 2     
            for threshold in thresholds:                                          
                left_indices = np.where(X[:, feature_index] <= threshold)[0]      
                right_indices = np.where(X[:, feature_index] > threshold)[0]      

                if (len(left_indices) < self.min_samples_leaf or 
                    len(right_indices) < self.min_samples_leaf):
                    continue                                                      

                if self.criterion == 'custom_1':
                    if y_oh is None:
                        raise ValueError("y_oh required for custom_1 criterion")
                    gain = self.custom_1(y_oh, left_indices, right_indices)
                
                elif self.criterion == 'custom_2':
                    if y_oh is None:
                        raise ValueError("y_oh required for custom_2 criterion")
                    gain = self.custom_2(y_oh, left_indices, right_indices)
                
                elif self.criterion == 'custom_3':
                    if y_oh is None:
                        raise ValueError("y_oh required for custom_3 criterion")
                    gain = self.custom_3(y_oh, left_indices, right_indices)                    
                
                elif self.criterion == 'custom_4':
                    if y_oh is None:
                        raise ValueError("y_oh required for custom_4 criterion")
                    gain = self.custom_4(y_oh, left_indices, right_indices)
                
                elif self.criterion == 'custom_5':
                    if y_oh is None:
                        raise ValueError("y_oh required for custom_5 criterion")
                    gain = self.custom_5(y_oh, left_indices, right_indices)
                
                elif self.criterion == 'custom_6':
                    if y_oh is None:
                        raise ValueError("y_oh required for custom_6 criterion")
                    gain = self.custom_6(y_oh, left_indices, right_indices)    
                    
                elif self.criterion == 'custom_7':
                    if y_oh is None:
                        raise ValueError("y_oh required for custom_7 criterion")
                    gain = self.custom_7(y_oh, left_indices, right_indices)
                    
                elif self.criterion == 'custom_8':
                    if y_oh is None:
                        raise ValueError("y_oh required for custom_7 criterion")
                    gain = self.custom_8(y_oh, left_indices, right_indices)                      
                
                else:
                    gain = self.information_gain(y, left_indices, right_indices)  

                if gain > best_gain:                                               
                    best_gain = gain                                               
                    best_split = {
                        'feature_index': feature_index,
                        'threshold': threshold,
                        'left_indices': left_indices,
                        'right_indices': right_indices,
                        'gain': gain
                    }                                                              
        
        return best_split                                                          


    def fit(self, X, y, y_oh=None):
        num_features = X.shape[1]
        self.feature_importances = np.zeros(num_features)                          
        self.tree = self.grow_tree(X, y, y_oh, depth=0)
        
        total = self.feature_importances.sum()
        if total > 0:
            self.feature_importances /= total


    def grow_tree(self, X, y, y_oh, depth):
        num_samples, num_features = X.shape
        num_classes = len(set(y))

        if (depth == self.max_depth or 
            num_classes == 1 or 
            num_samples < self.min_samples_split):
            return self.most_common_label(y)

        if self.criterion.startswith('custom_'):
            best_split = self.find_best_split(X, y, num_features, y_oh)
        else:
            best_split = self.find_best_split(X, y, num_features)

        if best_split is None:
            return self.most_common_label(y)

        left_indices, right_indices = best_split['left_indices'], best_split['right_indices']
        
        
        if self.criterion == 'custom_1':
            gain = self.custom_1(y_oh, left_indices, right_indices)
        elif self.criterion == 'custom_2':
            gain = self.custom_2(y_oh, left_indices, right_indices)
        elif self.criterion == 'custom_3':
            gain = self.custom_3(y_oh, left_indices, right_indices)
        elif self.criterion == 'custom_4':
            gain = self.custom_4(y_oh, left_indices, right_indices)
        elif self.criterion == 'custom_5':
            gain = self.custom_5(y_oh, left_indices, right_indices)
        elif self.criterion == 'custom_6':
            gain = self.custom_6(y_oh, left_indices, right_indices)
        elif self.criterion == 'custom_7':
            gain = self.custom_7(y_oh, left_indices, right_indices)
        elif self.criterion == 'custom_8':
            gain = self.custom_8(y_oh, left_indices, right_indices)            
            
        else:
            gain = self.information_gain(y, left_indices, right_indices)

        self.feature_importances[best_split['feature_index']] += gain              

        left_subtree = self.grow_tree(X[left_indices], y[left_indices], 
                                    y_oh[left_indices] if y_oh is not None else None, 
                                    depth + 1)
        right_subtree = self.grow_tree(X[right_indices], y[right_indices], 
                                     y_oh[right_indices] if y_oh is not None else None, 
                                     depth + 1)

        return {'feature_index': best_split['feature_index'],
                'threshold': best_split['threshold'],
                'left': left_subtree,
                'right': right_subtree}


    def predict(self, X):
        return np.array([self._traverse_tree(x, self.tree) for x in X])


    def _traverse_tree(self, x, node):
        if isinstance(node, dict):
            if x[node['feature_index']] <= node['threshold']:
                return self._traverse_tree(x, node['left'])
            else:
                return self._traverse_tree(x, node['right'])

        return node                                              

### Mean/std of 50 exps.

In [ ]:
# synthetic datsets
def compare_metrics_generated_datasets(N, V, k, alpha, nmin, max_depth,
                                       n_datasets=50,test_size=0.25,
                                       sig_range=(0.05, 0.10),n_jobs=-1):

    criteria = ["entropy_sklearn","custom_1","custom_2","custom_3","custom_4","custom_5","custom_6","custom_7","custom_8"]

    columns = ["entropy_sklearn",
               "b = 1",
               "b = p_l ^ 0.5",
               "b = (p_l*(1 - p_l)) ^ 0.5",
               "b = p_l",
               "b = p_l ^ 2",
               "b = -log(p_l)",
               "b = -p_l^0.5 * log(p_l)",
               "b = -p_l * log(p_l)"]

    def run_dataset(seed):
        Nk, R, y, X, cen = generdat(N=N,V=V,k=k,alpha=alpha,nmin=nmin,seed=seed,sig_range=sig_range)
        X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=test_size,random_state=seed,stratify=y)
        y_oh_train = np.eye(k, dtype=float)[y_train]

        dataset_results = []

        for criterion in criteria:
            if criterion == "entropy_sklearn":
                model = DecisionTreeClassifier(max_depth=max_depth,min_samples_split=2,min_samples_leaf=1,criterion="entropy",random_state=seed)
                model.fit(X_train, y_train)
            else:
                model = DecisionTree(max_depth=max_depth,min_samples_split=2,min_samples_leaf=1,criterion=criterion)
                model.fit(X_train, y_train, y_oh_train)

            y_pred = model.predict(X_test)
            scores = [accuracy_score(y_test, y_pred),precision_score(y_test,y_pred,average="weighted",zero_division=0),
                      recall_score(y_test,y_pred,average="weighted",zero_division=0),f1_score(y_test,y_pred,average="weighted",zero_division=0),
                      adjusted_rand_score(y_test, y_pred)]
            
            dataset_results.append(scores)

        return np.asarray(dataset_results).T

    all_results = Parallel(n_jobs=n_jobs)(delayed(run_dataset)(seed)for seed in range(1, n_datasets + 1))
    all_results = np.asarray(all_results)
    mean_results = np.round(all_results.mean(axis=0), 4)
    std_results = np.round(all_results.std(axis=0), 4)

    metrics = ["Accuracy","Precision","Recall","F1 score","ARI"]
    index = []
    table_data = []

    for metric_index, metric in enumerate(metrics):
        index.append((metric, "Mean"))
        table_data.append(mean_results[metric_index])
        index.append((metric, "Std"))
        table_data.append(std_results[metric_index])

    index = pd.MultiIndex.from_tuples(index,names=["Metric", "Statistic"])
    final_table = pd.DataFrame(table_data,columns=columns,index=index)    
    print("\n"f"N, V, k, alpha, nmin, max_depth, n_datasets = "f"{N, V, k, alpha, nmin, max_depth, n_datasets}")

    return final_table

In [ ]:
# Real World datasets exp.

def compare_metrics_train_test(max_depth, X, y,*, N=None, V=None, k=None, alpha=None, nmin=None, n_jobs=-1):

    def run_seed(seed):

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=seed)
        encoder = OneHotEncoder(sparse_output=False)
        y_oh_train = encoder.fit_transform(y_train.reshape(-1, 1))

        custom_1 = DecisionTree(max_depth=max_depth, criterion='custom_1')
        custom_1.fit(X_train, y_train, y_oh_train)
        y_pred = custom_1.predict(X_test)
        accuracy_1, precision_1 = accuracy_score(y_test, y_pred), precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall_1, f1_1 = recall_score(y_test, y_pred, average='weighted', zero_division=0), f1_score(y_test, y_pred, average='weighted')
        ari_1 = adjusted_rand_score(y_test, y_pred)

        sk_entropy = DecisionTreeClassifier(max_depth=max_depth, criterion='entropy', random_state=42)
        sk_entropy.fit(X_train, y_train)
        y_pred = sk_entropy.predict(X_test)
        accuracy_entropy_sk, precision_entropy_sk = accuracy_score(y_test, y_pred), precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall_entropy_sk, f1_entropy_sk = recall_score(y_test, y_pred, average='weighted', zero_division=0), f1_score(y_test, y_pred, average='weighted')
        ari_entropy_sk = adjusted_rand_score(y_test, y_pred)

        custom_2 = DecisionTree(max_depth=max_depth, criterion='custom_2')
        custom_2.fit(X_train, y_train, y_oh_train)
        y_pred = custom_2.predict(X_test)
        accuracy_2, precision_2 = accuracy_score(y_test, y_pred), precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall_2, f1_2 = recall_score(y_test, y_pred, average='weighted', zero_division=0), f1_score(y_test, y_pred, average='weighted')
        ari_2 = adjusted_rand_score(y_test, y_pred)

        custom_3 = DecisionTree(max_depth=max_depth, criterion='custom_3')
        custom_3.fit(X_train, y_train, y_oh_train)
        y_pred = custom_3.predict(X_test)
        accuracy_3, precision_3 = accuracy_score(y_test, y_pred), precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall_3, f1_3 = recall_score(y_test, y_pred, average='weighted', zero_division=0), f1_score(y_test, y_pred, average='weighted')
        ari_3 = adjusted_rand_score(y_test, y_pred)

        custom_4 = DecisionTree(max_depth=max_depth, criterion='custom_4')
        custom_4.fit(X_train, y_train, y_oh_train)
        y_pred = custom_4.predict(X_test)
        accuracy_4, precision_4 = accuracy_score(y_test, y_pred), precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall_4, f1_4 = recall_score(y_test, y_pred, average='weighted', zero_division=0), f1_score(y_test, y_pred, average='weighted')
        ari_4 = adjusted_rand_score(y_test, y_pred)

        custom_5 = DecisionTree(max_depth=max_depth, criterion='custom_5')
        custom_5.fit(X_train, y_train, y_oh_train)
        y_pred = custom_5.predict(X_test)
        accuracy_5, precision_5 = accuracy_score(y_test, y_pred), precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall_5, f1_5 = recall_score(y_test, y_pred, average='weighted', zero_division=0), f1_score(y_test, y_pred, average='weighted')
        ari_5 = adjusted_rand_score(y_test, y_pred)

        custom_6 = DecisionTree(max_depth=max_depth, criterion='custom_6')
        custom_6.fit(X_train, y_train, y_oh_train)
        y_pred = custom_6.predict(X_test)
        accuracy_6, precision_6 = accuracy_score(y_test, y_pred), precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall_6, f1_6 = recall_score(y_test, y_pred, average='weighted', zero_division=0), f1_score(y_test, y_pred, average='weighted')
        ari_6 = adjusted_rand_score(y_test, y_pred)

        custom_7 = DecisionTree(max_depth=max_depth, criterion='custom_7')
        custom_7.fit(X_train, y_train, y_oh_train)
        y_pred = custom_7.predict(X_test)
        accuracy_7, precision_7 = accuracy_score(y_test, y_pred), precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall_7, f1_7 = recall_score(y_test, y_pred, average='weighted', zero_division=0), f1_score(y_test, y_pred, average='weighted')
        ari_7 = adjusted_rand_score(y_test, y_pred)

        custom_8 = DecisionTree(max_depth=max_depth, criterion='custom_8')
        custom_8.fit(X_train, y_train, y_oh_train)
        y_pred = custom_8.predict(X_test)
        accuracy_8, precision_8 = accuracy_score(y_test, y_pred), precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall_8, f1_8 = recall_score(y_test, y_pred, average='weighted', zero_division=0), f1_score(y_test, y_pred, average='weighted')
        ari_8 = adjusted_rand_score(y_test, y_pred)

        return np.round([[accuracy_entropy_sk, accuracy_1, accuracy_2,accuracy_3, accuracy_4, accuracy_5,accuracy_6, accuracy_7, accuracy_8],
                         [precision_entropy_sk, precision_1, precision_2,precision_3, precision_4, precision_5,precision_6, precision_7, precision_8],
                         [recall_entropy_sk, recall_1, recall_2,recall_3, recall_4, recall_5,recall_6, recall_7, recall_8],
                         [f1_entropy_sk, f1_1, f1_2,f1_3, f1_4, f1_5,f1_6, f1_7, f1_8],
                         [ari_entropy_sk, ari_1, ari_2,ari_3, ari_4, ari_5,ari_6, ari_7, ari_8]], 4)

    all_results = Parallel(n_jobs=n_jobs)(delayed(run_seed)(seed) for seed in range(1, 51))

    print(f'\nN, V, k, alpha, nmin, max_depth = {N, V, k, alpha, nmin, max_depth}')

    all_results = np.array(all_results)
    mean_results = np.round(np.mean(all_results, axis=0), 4)
    std_results = np.round(np.std(all_results, axis=0), 4)

    columns = ['entropy_sklearn',
               'b = 1',
               'b = p_l ^ 0.5',
               'b = (p_l*(1 - p_l)) ^ 0.5',
               'b = p_l',
               'b = p_l ^ 2',
               'b = -log(p_l)',
               'b = -p_l^0.5 * log(p_l)',     
               'b = -p_l * log(p_l)']
    
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1 score', 'ARI']
    index_tuples = []

    for metric in metrics:
        index_tuples.append((metric, 'Mean'))
        index_tuples.append((metric, 'Std'))

    multi_index = pd.MultiIndex.from_tuples(index_tuples, names=['Metric', 'Statistic'])

    final_table_data = []

    for i in range(len(metrics)):
        final_table_data.append(mean_results[i])
        final_table_data.append(std_results[i])

    return pd.DataFrame(final_table_data,columns=columns,index=multi_index)

# 2. Experiments with Generated and Real World Datasets

---
## Generated Datasets

#### Data generator

Parameters:
- N: Total number of data points
- V: Number of dimensions/features
- k: Number of clusters
- alpha: Controls cluster center spread (centers are in [α-1, 1-α])
- nmin: Minimum points per cluster
- seed: Random seed for reproducibility
- sig_range: Tuple (min, max) for cluster standard deviations

Returns:
- Nk: Array of cluster sizes
- R: List of ranges for each cluster
- y: Cluster labels for each point
- X: Generated data (N x V array)
- cen: Cluster centers (k x V array)

In [ ]:
def generdat(N, V, k, alpha, nmin, seed=None, sig_range=(0.05, 0.1)):
    if N < k * nmin:
        raise ValueError(f"N must be >= k * nmin. Got N={N}, k={k}, nmin={nmin}")
    if k < 1:
        raise ValueError("k must be at least 1")
    if alpha == 1:
        raise ValueError("alpha cannot be 1")
    if seed is not None:
        np.random.seed(seed)

    if k == 1:
        Nk = np.array([N])
    else:
        base_sizes = np.ones(k, dtype=int) * nmin
        remaining = N - k * nmin
        if remaining > 0:
            additional = np.random.multinomial(remaining, np.ones(k)/k)
            Nk = base_sizes + additional
        else:
            Nk = base_sizes

    cen = (alpha - 1) + 2 * (1 - alpha) * np.random.rand(k, V)
    X = np.zeros((N, V))
    y = np.zeros(N, dtype=int)
    R = []
    
    sig_min, sig_max = sig_range
    start_idx = 0
    
    for k0 in range(k):
        nk = Nk[k0]
        end_idx = start_idx + nk
        
        R.append(range(start_idx, end_idx))
        y[start_idx:end_idx] = k0 
        
        sig = sig_min + (sig_max - sig_min) * np.random.rand(V)
        X[start_idx:end_idx] = np.random.randn(nk, V) * sig + cen[k0, :]
        
        start_idx = end_idx

    return Nk, R, y, X, cen

### Generated Dataset / Tree_depth = 3

In [ ]:
tables = []

for cluster in [4,8,15]:
    for feature in [6,15]:
        for squeeze in [0.5, 0.85]:
            N, V, k, alpha, nmin = 2000, feature, cluster, squeeze, 50
            table = compare_metrics_generated_datasets(N=N,V=V,k=k,alpha=alpha,nmin=nmin,max_depth=3)
            tables.append(table)


N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 6, 4, 0.5, 50, 3, 50)

N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 6, 4, 0.85, 50, 3, 50)

N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 15, 4, 0.5, 50, 3, 50)

N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 15, 4, 0.85, 50, 3, 50)

N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 6, 8, 0.5, 50, 3, 50)

N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 6, 8, 0.85, 50, 3, 50)

N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 15, 8, 0.5, 50, 3, 50)

N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 15, 8, 0.85, 50, 3, 50)

N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 6, 15, 0.5, 50, 3, 50)

N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 6, 15, 0.85, 50, 3, 50)

N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 15, 15, 0.5, 50, 3, 50)

N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 15, 15, 0.85, 50, 3, 50)


In [136]:
# N, V, k, alpha, nmin, max_depth = (2000, 6, 4, 0.5, 50, 3)
tables[0]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.9949  0.9937         0.9732   
          Std                 0.0158  0.0184         0.0733   
Precision Mean                0.9949  0.9938         0.9688   
          Std                 0.0158  0.0181         0.0926   
Recall    Mean                0.9949  0.9937         0.9732   
          Std                 0.0158  0.0184         0.0733   
F1 score  Mean                0.9949  0.9937         0.9654   
          Std                 0.0159  0.0183         0.0967   
ARI       Mean                0.9875  0.9845         0.9643   
          Std                 0.0351  0.0412         0.0890   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.9912   0.8742       0.8526   
          Std                           0.0362   0.1505       0.1339   
Precision Mean                          0.9920   0.8748       0.8463   
          Std                           0.0305   0.1696       0.1653   
Recall    Mean                          0.9912   0.8742       0.8526   
          Std                           0.0362   0.1505       0.1339   
F1 score  Mean                          0.9898   0.8358       0.8067   
          Std                           0.0457   0.1946       0.1745   
ARI       Mean                          0.9836   0.8440       0.8169   
          Std                           0.0494   0.1880       0.1676   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.9924                   0.9935   
          Std               0.0190                   0.0183   
Precision Mean              0.9926                   0.9936   
          Std               0.0187                   0.0181   
Recall    Mean              0.9924                   0.9935   
          Std               0.0190                   0.0183   
F1 score  Mean              0.9925                   0.9935   
          Std               0.0189                   0.0182   
ARI       Mean              0.9813                   0.9841   
          Std               0.0429                   0.0410   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.9490  
          Std                     0.0971  
Precision Mean                    0.9274  
          Std                     0.1421  
Recall    Mean                    0.9490  
          Std                     0.0971  
F1 score  Mean                    0.9327  
          Std                     0.1297  
ARI       Mean                    0.9360  
          Std                     0.1152

In [ ]:
# N, V, k, alpha, nmin, max_depth = (2000, 6, 4, 0.85, 50, 3)
tables[1]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.7804  0.7828         0.7714   
          Std                 0.0686  0.0646         0.0752   
Precision Mean                0.7912  0.7938         0.7751   
          Std                 0.0654  0.0623         0.0841   
Recall    Mean                0.7804  0.7828         0.7714   
          Std                 0.0686  0.0646         0.0752   
F1 score  Mean                0.7785  0.7829         0.7622   
          Std                 0.0720  0.0651         0.0884   
ARI       Mean                0.5261  0.5294         0.5301   
          Std                 0.1173  0.1111         0.1156   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.7806   0.5241       0.5155   
          Std                           0.0692   0.0471       0.0445   
Precision Mean                          0.7855   0.4978       0.4750   
          Std                           0.0783   0.1042       0.1029   
Recall    Mean                          0.7806   0.5241       0.5155   
          Std                           0.0692   0.0471       0.0445   
F1 score  Mean                          0.7768   0.4309       0.4277   
          Std                           0.0769   0.0681       0.0652   
ARI       Mean                          0.5351   0.3093       0.2918   
          Std                           0.1094   0.0687       0.0695   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.7853                   0.7842   
          Std               0.0648                   0.0643   
Precision Mean              0.7961                   0.7953   
          Std               0.0611                   0.0621   
Recall    Mean              0.7853                   0.7842   
          Std               0.0648                   0.0643   
F1 score  Mean              0.7858                   0.7841   
          Std               0.0645                   0.0654   
ARI       Mean              0.5316                   0.5300   
          Std               0.1131                   0.1123   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.7740  
          Std                     0.0683  
Precision Mean                    0.7814  
          Std                     0.0788  
Recall    Mean                    0.7740  
          Std                     0.0683  
F1 score  Mean                    0.7671  
          Std                     0.0789  
ARI       Mean                    0.5269  
          Std                     0.1081

In [ ]:
# N, V, k, alpha, nmin, max_depth = (2000, 15, 4, 0.5, 50, 3)
tables[2]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.9986  0.9980         0.9944   
          Std                 0.0022  0.0029         0.0327   
Precision Mean                0.9986  0.9980         0.9919   
          Std                 0.0022  0.0029         0.0499   
Recall    Mean                0.9986  0.9980         0.9944   
          Std                 0.0022  0.0029         0.0327   
F1 score  Mean                0.9986  0.9980         0.9927   
          Std                 0.0022  0.0029         0.0439   
ARI       Mean                0.9963  0.9946         0.9918   
          Std                 0.0058  0.0076         0.0390   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.9981   0.9433       0.9124   
          Std                           0.0028   0.1163       0.1204   
Precision Mean                          0.9981   0.9302       0.8837   
          Std                           0.0028   0.1424       0.1705   
Recall    Mean                          0.9981   0.9433       0.9124   
          Std                           0.0028   0.1163       0.1204   
F1 score  Mean                          0.9981   0.9257       0.8837   
          Std                           0.0028   0.1515       0.1605   
ARI       Mean                          0.9950   0.9284       0.8928   
          Std                           0.0074   0.1467       0.1453   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.9972                   0.9976   
          Std               0.0040                   0.0032   
Precision Mean              0.9972                   0.9977   
          Std               0.0039                   0.0032   
Recall    Mean              0.9972                   0.9976   
          Std               0.0040                   0.0032   
F1 score  Mean              0.9972                   0.9976   
          Std               0.0040                   0.0032   
ARI       Mean              0.9925                   0.9937   
          Std               0.0106                   0.0085   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.9565  
          Std                     0.0901  
Precision Mean                    0.9397  
          Std                     0.1299  
Recall    Mean                    0.9565  
          Std                     0.0901  
F1 score  Mean                    0.9424  
          Std                     0.1203  
ARI       Mean                    0.9475  
          Std                     0.1048

In [ ]:
# N, V, k, alpha, nmin, max_depth = (2000, 15, 4, 0.85, 50, 3)
tables[3]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.8748  0.8774         0.8581   
          Std                 0.0384  0.0376         0.0594   
Precision Mean                0.8806  0.8839         0.8615   
          Std                 0.0363  0.0340         0.0658   
Recall    Mean                0.8748  0.8774         0.8581   
          Std                 0.0384  0.0376         0.0594   
F1 score  Mean                0.8752  0.8783         0.8529   
          Std                 0.0383  0.0371         0.0738   
ARI       Mean                0.7005  0.7045         0.6839   
          Std                 0.0825  0.0813         0.0932   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.8752   0.5033       0.5055   
          Std                           0.0398   0.0189       0.0257   
Precision Mean                          0.8813   0.4920       0.5028   
          Std                           0.0366   0.1418       0.1278   
Recall    Mean                          0.8752   0.5033       0.5055   
          Std                           0.0398   0.0189       0.0257   
F1 score  Mean                          0.8759   0.3876       0.3973   
          Std                           0.0395   0.0338       0.0441   
ARI       Mean                          0.7009   0.3114       0.3065   
          Std                           0.0854   0.0559       0.0575   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.8740                   0.8757   
          Std               0.0380                   0.0368   
Precision Mean              0.8807                   0.8820   
          Std               0.0353                   0.0341   
Recall    Mean              0.8740                   0.8757   
          Std               0.0380                   0.0368   
F1 score  Mean              0.8750                   0.8766   
          Std               0.0376                   0.0364   
ARI       Mean              0.6969                   0.7004   
          Std               0.0817                   0.0793   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.8544  
          Std                     0.0703  
Precision Mean                    0.8622  
          Std                     0.0727  
Recall    Mean                    0.8544  
          Std                     0.0703  
F1 score  Mean                    0.8505  
          Std                     0.0873  
ARI       Mean                    0.6726  
          Std                     0.1011

In [ ]:
# N, V, k, alpha, nmin, max_depth = (2000, 6, 8, 0.5, 50, 3)
tables[4]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.9056  0.7183         0.6600   
          Std                 0.0578  0.1192         0.1488   
Precision Mean                0.8742  0.6361         0.5567   
          Std                 0.0881  0.1553         0.1757   
Recall    Mean                0.9056  0.7183         0.6600   
          Std                 0.0578  0.1192         0.1488   
F1 score  Mean                0.8802  0.6436         0.5711   
          Std                 0.0777  0.1440         0.1777   
ARI       Mean                0.8765  0.6656         0.6070   
          Std                 0.0645  0.1581         0.1954   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.7268   0.5128       0.4734   
          Std                           0.1124   0.1656       0.1194   
Precision Mean                          0.6328   0.4354       0.4037   
          Std                           0.1487   0.1690       0.1315   
Recall    Mean                          0.7268   0.5128       0.4734   
          Std                           0.1124   0.1656       0.1194   
F1 score  Mean                          0.6525   0.4096       0.3631   
          Std                           0.1390   0.1854       0.1341   
ARI       Mean                          0.6740   0.4197       0.3770   
          Std                           0.1434   0.2135       0.1618   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.7038                   0.7200   
          Std               0.1219                   0.1153   
Precision Mean              0.6366                   0.6440   
          Std               0.1548                   0.1502   
Recall    Mean              0.7038                   0.7200   
          Std               0.1219                   0.1153   
F1 score  Mean              0.6278                   0.6459   
          Std               0.1470                   0.1395   
ARI       Mean              0.6451                   0.6676   
          Std               0.1649                   0.1536   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.5694  
          Std                     0.1502  
Precision Mean                    0.4842  
          Std                     0.1704  
Recall    Mean                    0.5694  
          Std                     0.1502  
F1 score  Mean                    0.4699  
          Std                     0.1750  
ARI       Mean                    0.4895  
          Std                     0.1924

In [ ]:
# N, V, k, alpha, nmin, max_depth = (2000, 6, 8, 0.85, 50, 3)
tables[5]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.5639  0.5518         0.5150   
          Std                 0.0552  0.0667         0.0671   
Precision Mean                0.5172  0.5211         0.4289   
          Std                 0.0806  0.0901         0.0862   
Recall    Mean                0.5639  0.5518         0.5150   
          Std                 0.0552  0.0667         0.0671   
F1 score  Mean                0.5210  0.5091         0.4400   
          Std                 0.0676  0.0794         0.0801   
ARI       Mean                0.3433  0.3244         0.3236   
          Std                 0.0640  0.0710         0.0687   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.5396   0.2947       0.2945   
          Std                           0.0687   0.0314       0.0343   
Precision Mean                          0.4854   0.2233       0.2253   
          Std                           0.0867   0.0736       0.0766   
Recall    Mean                          0.5396   0.2947       0.2945   
          Std                           0.0687   0.0314       0.0343   
F1 score  Mean                          0.4813   0.1817       0.1929   
          Std                           0.0840   0.0352       0.0389   
ARI       Mean                          0.3316   0.1565       0.1465   
          Std                           0.0738   0.0370       0.0381   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.5498                   0.5520   
          Std               0.0650                   0.0661   
Precision Mean              0.5217                   0.5241   
          Std               0.0801                   0.0808   
Recall    Mean              0.5498                   0.5520   
          Std               0.0650                   0.0661   
F1 score  Mean              0.5080                   0.5106   
          Std               0.0748                   0.0755   
ARI       Mean              0.3162                   0.3223   
          Std               0.0728                   0.0715   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.4697  
          Std                     0.0769  
Precision Mean                    0.3800  
          Std                     0.1095  
Recall    Mean                    0.4697  
          Std                     0.0769  
F1 score  Mean                    0.3793  
          Std                     0.0934  
ARI       Mean                    0.2865  
          Std                     0.0754

In [ ]:
# N, V, k, alpha, nmin, max_depth = (2000, 15, 8, 0.5, 50, 3)
tables[6]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.9568  0.6850         0.6879   
          Std                 0.0516  0.1053         0.1103   
Precision Mean                0.9388  0.5883         0.5727   
          Std                 0.0793  0.1265         0.1464   
Recall    Mean                0.9568  0.6850         0.6879   
          Std                 0.0516  0.1053         0.1103   
F1 score  Mean                0.9448  0.6052         0.6028   
          Std                 0.0697  0.1247         0.1357   
ARI       Mean                0.9453  0.6202         0.6330   
          Std                 0.0531  0.1431         0.1433   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.7172   0.4886       0.4218   
          Std                           0.0939   0.1399       0.0929   
Precision Mean                          0.6104   0.3922       0.3573   
          Std                           0.1232   0.1456       0.1079   
Recall    Mean                          0.7172   0.4886       0.4218   
          Std                           0.0939   0.1399       0.0929   
F1 score  Mean                          0.6396   0.3853       0.3219   
          Std                           0.1145   0.1537       0.0951   
ARI       Mean                          0.6654   0.3680       0.2603   
          Std                           0.1212   0.1783       0.1251   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.6469                   0.6776   
          Std               0.1072                   0.1043   
Precision Mean              0.5717                   0.5822   
          Std               0.1273                   0.1221   
Recall    Mean              0.6469                   0.6776   
          Std               0.1072                   0.1043   
F1 score  Mean              0.5688                   0.5969   
          Std               0.1246                   0.1230   
ARI       Mean              0.5420                   0.6097   
          Std               0.1493                   0.1413   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.6642  
          Std                     0.1128  
Precision Mean                    0.5454  
          Std                     0.1423  
Recall    Mean                    0.6642  
          Std                     0.1128  
F1 score  Mean                    0.5757  
          Std                     0.1344  
ARI       Mean                    0.6081  
          Std                     0.1450

In [ ]:
# N, V, k, alpha, nmin, max_depth = (2000, 15, 8, 0.85, 50, 3)
tables[7]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.6706  0.6427         0.5803   
          Std                 0.0392  0.0583         0.0751   
Precision Mean                0.6412  0.6095         0.4898   
          Std                 0.0719  0.0803         0.1124   
Recall    Mean                0.6706  0.6427         0.5803   
          Std                 0.0392  0.0583         0.0751   
F1 score  Mean                0.6381  0.6016         0.5023   
          Std                 0.0543  0.0728         0.0983   
ARI       Mean                0.4695  0.4411         0.4109   
          Std                 0.0428  0.0619         0.0698   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.6296   0.2702       0.2844   
          Std                           0.0619   0.0267       0.0240   
Precision Mean                          0.5949   0.2123       0.2340   
          Std                           0.0958   0.0942       0.0767   
Recall    Mean                          0.6296   0.2702       0.2844   
          Std                           0.0619   0.0267       0.0240   
F1 score  Mean                          0.5803   0.1470       0.1783   
          Std                           0.0805   0.0325       0.0323   
ARI       Mean                          0.4333   0.1479       0.1482   
          Std                           0.0665   0.0298       0.0353   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.6356                   0.6406   
          Std               0.0587                   0.0565   
Precision Mean              0.6038                   0.6049   
          Std               0.0732                   0.0770   
Recall    Mean              0.6356                   0.6406   
          Std               0.0587                   0.0565   
F1 score  Mean              0.5952                   0.5993   
          Std               0.0732                   0.0717   
ARI       Mean              0.4323                   0.4388   
          Std               0.0617                   0.0614   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.4637  
          Std                     0.0839  
Precision Mean                    0.3624  
          Std                     0.1047  
Recall    Mean                    0.4637  
          Std                     0.0839  
F1 score  Mean                    0.3582  
          Std                     0.1056  
ARI       Mean                    0.3089  
          Std                     0.0769

In [ ]:
# N, V, k, alpha, nmin, max_depth = (2000, 6, 15, 0.5, 50, 3)
tables[8]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.5365  0.4661         0.4130   
          Std                 0.0116  0.0641         0.0776   
Precision Mean                0.3339  0.3349         0.2770   
          Std                 0.0153  0.0605         0.0694   
Recall    Mean                0.5365  0.4661         0.4130   
          Std                 0.0116  0.0641         0.0776   
F1 score  Mean                0.3989  0.3585         0.2931   
          Std                 0.0099  0.0643         0.0764   
ARI       Mean                0.5553  0.3749         0.3560   
          Std                 0.0410  0.1115         0.1092   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.4752   0.2081       0.2372   
          Std                           0.0624   0.0575       0.0470   
Precision Mean                          0.3321   0.1510       0.1743   
          Std                           0.0553   0.0605       0.0486   
Recall    Mean                          0.4752   0.2081       0.2372   
          Std                           0.0624   0.0575       0.0470   
F1 score  Mean                          0.3631   0.1136       0.1485   
          Std                           0.0584   0.0515       0.0428   
ARI       Mean                          0.4026   0.1218       0.1197   
          Std                           0.1144   0.0714       0.0617   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.4513                   0.4665   
          Std               0.0656                   0.0651   
Precision Mean              0.3370                   0.3337   
          Std               0.0585                   0.0655   
Recall    Mean              0.4513                   0.4665   
          Std               0.0656                   0.0651   
F1 score  Mean              0.3464                   0.3587   
          Std               0.0659                   0.0667   
ARI       Mean              0.3569                   0.3742   
          Std               0.1081                   0.1101   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.2844  
          Std                     0.0868  
Precision Mean                    0.1976  
          Std                     0.0775  
Recall    Mean                    0.2844  
          Std                     0.0868  
F1 score  Mean                    0.1776  
          Std                     0.0828  
ARI       Mean                    0.2065  
          Std                     0.1040

In [ ]:
# N, V, k, alpha, nmin, max_depth = (2000, 6, 15, 0.85, 50, 3)
tables[9]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.3658  0.3578         0.3253   
          Std                 0.0326  0.0310         0.0342   
Precision Mean                0.2125  0.2340         0.1794   
          Std                 0.0264  0.0278         0.0432   
Recall    Mean                0.3658  0.3578         0.3253   
          Std                 0.0326  0.0310         0.0342   
F1 score  Mean                0.2596  0.2612         0.2141   
          Std                 0.0280  0.0254         0.0380   
ARI       Mean                0.2277  0.2021         0.1982   
          Std                 0.0355  0.0415         0.0358   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.3369   0.1676       0.1794   
          Std                           0.0336   0.0228       0.0232   
Precision Mean                          0.1963   0.0868       0.0931   
          Std                           0.0427   0.0362       0.0355   
Recall    Mean                          0.3369   0.1676       0.1794   
          Std                           0.0336   0.0228       0.0232   
F1 score  Mean                          0.2298   0.0735       0.0937   
          Std                           0.0381   0.0201       0.0201   
ARI       Mean                          0.2016   0.0817       0.0883   
          Std                           0.0347   0.0224       0.0231   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.3564                   0.3574   
          Std               0.0351                   0.0317   
Precision Mean              0.2290                   0.2314   
          Std               0.0272                   0.0278   
Recall    Mean              0.3564                   0.3574   
          Std               0.0351                   0.0317   
F1 score  Mean              0.2592                   0.2611   
          Std               0.0288                   0.0275   
ARI       Mean              0.2008                   0.2018   
          Std               0.0426                   0.0410   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.2539  
          Std                     0.0421  
Precision Mean                    0.1241  
          Std                     0.0504  
Recall    Mean                    0.2539  
          Std                     0.0421  
F1 score  Mean                    0.1379  
          Std                     0.0419  
ARI       Mean                    0.1486  
          Std                     0.0368

In [ ]:
# N, V, k, alpha, nmin, max_depth = (2000, 15, 15, 0.5, 50, 3)
tables[10]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.5469  0.4609         0.4233   
          Std                 0.0075  0.0563         0.0776   
Precision Mean                0.3282  0.3388         0.2824   
          Std                 0.0141  0.0504         0.0806   
Recall    Mean                0.5469  0.4609         0.4233   
          Std                 0.0075  0.0563         0.0776   
F1 score  Mean                0.4013  0.3599         0.3083   
          Std                 0.0096  0.0569         0.0823   
ARI       Mean                0.6074  0.3527         0.3514   
          Std                 0.0217  0.0985         0.0962   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.4557   0.2424       0.2384   
          Std                           0.0611   0.0713       0.0484   
Precision Mean                          0.3139   0.1588       0.1647   
          Std                           0.0555   0.0653       0.0542   
Recall    Mean                          0.4557   0.2424       0.2384   
          Std                           0.0611   0.0713       0.0484   
F1 score  Mean                          0.3469   0.1450       0.1537   
          Std                           0.0584   0.0683       0.0434   
ARI       Mean                          0.3720   0.1464       0.1042   
          Std                           0.1044   0.0737       0.0627   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.4449                   0.4600   
          Std               0.0587                   0.0598   
Precision Mean              0.3469                   0.3339   
          Std               0.0468                   0.0578   
Recall    Mean              0.4449                   0.4600   
          Std               0.0587                   0.0598   
F1 score  Mean              0.3507                   0.3575   
          Std               0.0573                   0.0606   
ARI       Mean              0.3153                   0.3572   
          Std               0.0903                   0.0983   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.3077  
          Std                     0.0926  
Precision Mean                    0.2012  
          Std                     0.0815  
Recall    Mean                    0.3077  
          Std                     0.0926  
F1 score  Mean                    0.1983  
          Std                     0.0915  
ARI       Mean                    0.2334  
          Std                     0.0987

In [ ]:
# N, V, k, alpha, nmin, max_depth = (2000, 15, 15, 0.85, 50, 3)
tables[11]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.4146  0.4061         0.3410   
          Std                 0.0260  0.0267         0.0426   
Precision Mean                0.2462  0.2707         0.1825   
          Std                 0.0218  0.0234         0.0479   
Recall    Mean                0.4146  0.4061         0.3410   
          Std                 0.0260  0.0267         0.0426   
F1 score  Mean                0.2997  0.3041         0.2211   
          Std                 0.0227  0.0232         0.0468   
ARI       Mean                0.2822  0.2483         0.2224   
          Std                 0.0289  0.0363         0.0384   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.3733   0.1486       0.1838   
          Std                           0.0441   0.0134       0.0248   
Precision Mean                          0.2284   0.0881       0.1106   
          Std                           0.0559   0.0418       0.0374   
Recall    Mean                          0.3733   0.1486       0.1838   
          Std                           0.0441   0.0134       0.0248   
F1 score  Mean                          0.2639   0.0544       0.0982   
          Std                           0.0513   0.0137       0.0243   
ARI       Mean                          0.2334   0.0736       0.0892   
          Std                           0.0384   0.0118       0.0178   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.4020                   0.4072   
          Std               0.0271                   0.0269   
Precision Mean              0.2697                   0.2705   
          Std               0.0246                   0.0239   
Recall    Mean              0.4020                   0.4072   
          Std               0.0271                   0.0269   
F1 score  Mean              0.3018                   0.3048   
          Std               0.0238                   0.0243   
ARI       Mean              0.2430                   0.2489   
          Std               0.0386                   0.0363   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.2076  
          Std                     0.0390  
Precision Mean                    0.0979  
          Std                     0.0404  
Recall    Mean                    0.2076  
          Std                     0.0390  
F1 score  Mean                    0.0916  
          Std                     0.0313  
ARI       Mean                    0.1269  
          Std                     0.0376

### Generated Dataset / Tree_depth = 4

In [149]:
tables = []

for cluster in [4,8,15]:
    for feature in [6,15]:
        for squeeze in [0.5, 0.85]:
            N, V, k, alpha, nmin = 2000, feature, cluster, squeeze, 50
            table = compare_metrics_generated_datasets(N=N,V=V,k=k,alpha=alpha,nmin=nmin,max_depth=4)
            tables.append(table)


N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 6, 4, 0.5, 50, 4, 50)

N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 6, 4, 0.85, 50, 4, 50)

N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 15, 4, 0.5, 50, 4, 50)

N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 15, 4, 0.85, 50, 4, 50)

N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 6, 8, 0.5, 50, 4, 50)

N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 6, 8, 0.85, 50, 4, 50)

N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 15, 8, 0.5, 50, 4, 50)

N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 15, 8, 0.85, 50, 4, 50)

N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 6, 15, 0.5, 50, 4, 50)

N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 6, 15, 0.85, 50, 4, 50)

N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 15, 15, 0.5, 50, 4, 50)

N, V, k, alpha, nmin, max_depth, n_datasets = (2000, 15, 15, 0.85, 50, 4, 50)


In [ ]:
# N, V, k, alpha, nmin, max_depth = (2000, 6, 4, 0.5, 50, 4)
tables[0]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.9955  0.9952         0.9878   
          Std                 0.0146  0.0150         0.0480   
Precision Mean                0.9956  0.9952         0.9907   
          Std                 0.0145  0.0148         0.0346   
Recall    Mean                0.9955  0.9952         0.9878   
          Std                 0.0146  0.0150         0.0480   
F1 score  Mean                0.9955  0.9952         0.9849   
          Std                 0.0146  0.0149         0.0619   
ARI       Mean                0.9890  0.9879         0.9812   
          Std                 0.0326  0.0350         0.0613   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.9921   0.9636       0.9530   
          Std                           0.0355   0.0850       0.0946   
Precision Mean                          0.9929   0.9667       0.9496   
          Std                           0.0302   0.0853       0.1125   
Recall    Mean                          0.9921   0.9636       0.9530   
          Std                           0.0355   0.0850       0.0946   
F1 score  Mean                          0.9907   0.9535       0.9398   
          Std                           0.0452   0.1101       0.1232   
ARI       Mean                          0.9860   0.9521       0.9369   
          Std                           0.0467   0.1053       0.1198   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.9949                   0.9951   
          Std               0.0151                   0.0150   
Precision Mean              0.9949                   0.9951   
          Std               0.0148                   0.0148   
Recall    Mean              0.9949                   0.9951   
          Std               0.0151                   0.0150   
F1 score  Mean              0.9949                   0.9951   
          Std               0.0150                   0.0149   
ARI       Mean              0.9872                   0.9877   
          Std               0.0351                   0.0349   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.9787  
          Std                     0.0646  
Precision Mean                    0.9764  
          Std                     0.0775  
Recall    Mean                    0.9787  
          Std                     0.0646  
F1 score  Mean                    0.9725  
          Std                     0.0853  
ARI       Mean                    0.9711  
          Std                     0.0781

In [ ]:
# N, V, k, alpha, nmin, max_depth = (2000, 6, 4, 0.85, 50, 4)
tables[1]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.8058  0.8060         0.8003   
          Std                 0.0614  0.0594         0.0636   
Precision Mean                0.8114  0.8111         0.8028   
          Std                 0.0590  0.0589         0.0672   
Recall    Mean                0.8058  0.8060         0.8003   
          Std                 0.0614  0.0594         0.0636   
F1 score  Mean                0.8047  0.8054         0.7958   
          Std                 0.0623  0.0600         0.0722   
ARI       Mean                0.5734  0.5704         0.5687   
          Std                 0.1089  0.1068         0.1061   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.8066   0.5471       0.5536   
          Std                           0.0608   0.0584       0.0586   
Precision Mean                          0.8111   0.5804       0.5657   
          Std                           0.0603   0.0872       0.0892   
Recall    Mean                          0.8066   0.5471       0.5536   
          Std                           0.0608   0.0584       0.0586   
F1 score  Mean                          0.8045   0.4733       0.4919   
          Std                           0.0639   0.0824       0.0761   
ARI       Mean                          0.5758   0.3217       0.3157   
          Std                           0.1064   0.0703       0.0832   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.8046                   0.8064   
          Std               0.0619                   0.0605   
Precision Mean              0.8111                   0.8125   
          Std               0.0588                   0.0583   
Recall    Mean              0.8046                   0.8064   
          Std               0.0619                   0.0605   
F1 score  Mean              0.8045                   0.8062   
          Std               0.0616                   0.0603   
ARI       Mean              0.5661                   0.5700   
          Std               0.1126                   0.1097   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.7963  
          Std                     0.0631  
Precision Mean                    0.8011  
          Std                     0.0659  
Recall    Mean                    0.7963  
          Std                     0.0631  
F1 score  Mean                    0.7923  
          Std                     0.0693  
ARI       Mean                    0.5630  
          Std                     0.1050

In [ ]:
# N, V, k, alpha, nmin, max_depth = (2000, 15, 4, 0.5, 50, 4)
tables[2]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.9986  0.9987         0.9990   
          Std                 0.0022  0.0018         0.0015   
Precision Mean                0.9986  0.9987         0.9990   
          Std                 0.0022  0.0018         0.0014   
Recall    Mean                0.9986  0.9987         0.9990   
          Std                 0.0022  0.0018         0.0015   
F1 score  Mean                0.9986  0.9987         0.9990   
          Std                 0.0022  0.0018         0.0015   
ARI       Mean                0.9963  0.9965         0.9974   
          Std                 0.0058  0.0048         0.0039   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.9984   0.9886       0.9883   
          Std                           0.0026   0.0496       0.0493   
Precision Mean                          0.9984   0.9885       0.9881   
          Std                           0.0026   0.0561       0.0562   
Recall    Mean                          0.9984   0.9886       0.9883   
          Std                           0.0026   0.0496       0.0493   
F1 score  Mean                          0.9984   0.9853       0.9851   
          Std                           0.0026   0.0659       0.0650   
ARI       Mean                          0.9958   0.9843       0.9832   
          Std                           0.0068   0.0602       0.0614   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.9983                   0.9984   
          Std               0.0026                   0.0026   
Precision Mean              0.9983                   0.9984   
          Std               0.0025                   0.0026   
Recall    Mean              0.9983                   0.9984   
          Std               0.0026                   0.0026   
F1 score  Mean              0.9983                   0.9984   
          Std               0.0026                   0.0026   
ARI       Mean              0.9956                   0.9957   
          Std               0.0067                   0.0068   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.9940  
          Std                     0.0312  
Precision Mean                    0.9917  
          Std                     0.0473  
Recall    Mean                    0.9940  
          Std                     0.0312  
F1 score  Mean                    0.9924  
          Std                     0.0418  
ARI       Mean                    0.9906  
          Std                     0.0365

In [ ]:
# N, V, k, alpha, nmin, max_depth = (2000, 15, 4, 0.85, 50, 4)
tables[3]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.9034  0.9035         0.8901   
          Std                 0.0336  0.0293         0.0473   
Precision Mean                0.9059  0.9060         0.8929   
          Std                 0.0322  0.0281         0.0459   
Recall    Mean                0.9034  0.9035         0.8901   
          Std                 0.0336  0.0293         0.0473   
F1 score  Mean                0.9036  0.9036         0.8873   
          Std                 0.0336  0.0294         0.0566   
ARI       Mean                0.7636  0.7624         0.7437   
          Std                 0.0748  0.0655         0.0825   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.9020   0.5220       0.5612   
          Std                           0.0316   0.0414       0.0846   
Precision Mean                          0.9047   0.6059       0.6101   
          Std                           0.0308   0.0983       0.1059   
Recall    Mean                          0.9020   0.5220       0.5612   
          Std                           0.0316   0.0414       0.0846   
F1 score  Mean                          0.9019   0.4176       0.4810   
          Std                           0.0318   0.0599       0.1079   
ARI       Mean                          0.7597   0.3272       0.3678   
          Std                           0.0707   0.0694       0.1146   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.8997                   0.9007   
          Std               0.0330                   0.0322   
Precision Mean              0.9033                   0.9045   
          Std               0.0311                   0.0301   
Recall    Mean              0.8997                   0.9007   
          Std               0.0330                   0.0322   
F1 score  Mean              0.9000                   0.9011   
          Std               0.0329                   0.0321   
ARI       Mean              0.7532                   0.7557   
          Std               0.0742                   0.0724   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.8848  
          Std                     0.0655  
Precision Mean                    0.8923  
          Std                     0.0494  
Recall    Mean                    0.8848  
          Std                     0.0655  
F1 score  Mean                    0.8813  
          Std                     0.0822  
ARI       Mean                    0.7337  
          Std                     0.0914

In [ ]:
# N, V, k, alpha, nmin, max_depth = (2000, 6, 8, 0.5, 50, 4)
tables[4]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.9813  0.8988         0.8387   
          Std                 0.0191  0.1020         0.1451   
Precision Mean                0.9806  0.8667         0.8088   
          Std                 0.0257  0.1375         0.1655   
Recall    Mean                0.9813  0.8988         0.8387   
          Std                 0.0191  0.1020         0.1451   
F1 score  Mean                0.9806  0.8710         0.7978   
          Std                 0.0234  0.1333         0.1818   
ARI       Mean                0.9615  0.8767         0.7981   
          Std                 0.0301  0.1101         0.1787   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.9020   0.6319       0.5989   
          Std                           0.0878   0.1920       0.1607   
Precision Mean                          0.8750   0.6010       0.5809   
          Std                           0.1153   0.2040       0.1681   
Recall    Mean                          0.9020   0.6319       0.5989   
          Std                           0.0878   0.1920       0.1607   
F1 score  Mean                          0.8754   0.5490       0.5151   
          Std                           0.1146   0.2276       0.1846   
ARI       Mean                          0.8763   0.5520       0.4992   
          Std                           0.1017   0.2363       0.2111   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.8884                   0.9005   
          Std               0.1078                   0.0996   
Precision Mean              0.8710                   0.8705   
          Std               0.1370                   0.1315   
Recall    Mean              0.8884                   0.9005   
          Std               0.1078                   0.0996   
F1 score  Mean              0.8598                   0.8734   
          Std               0.1393                   0.1302   
ARI       Mean              0.8593                   0.8791   
          Std               0.1204                   0.1054   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.7221  
          Std                     0.1649  
Precision Mean                    0.6894  
          Std                     0.1776  
Recall    Mean                    0.7221  
          Std                     0.1649  
F1 score  Mean                    0.6542  
          Std                     0.2008  
ARI       Mean                    0.6596  
          Std                     0.2056

In [ ]:
# N, V, k, alpha, nmin, max_depth = (2000, 6, 8, 0.85, 50, 4)
tables[5]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.6254  0.6288         0.5907   
          Std                 0.0607  0.0627         0.0630   
Precision Mean                0.6421  0.6494         0.5751   
          Std                 0.0621  0.0654         0.0811   
Recall    Mean                0.6254  0.6288         0.5907   
          Std                 0.0607  0.0627         0.0630   
F1 score  Mean                0.6178  0.6229         0.5549   
          Std                 0.0641  0.0689         0.0745   
ARI       Mean                0.3809  0.3744         0.3727   
          Std                 0.0710  0.0750         0.0703   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.6160   0.3206       0.3255   
          Std                           0.0671   0.0446       0.0365   
Precision Mean                          0.6127   0.3001       0.3141   
          Std                           0.0836   0.0894       0.0961   
Recall    Mean                          0.6160   0.3206       0.3255   
          Std                           0.0671   0.0446       0.0365   
F1 score  Mean                          0.5954   0.2218       0.2410   
          Std                           0.0793   0.0511       0.0428   
ARI       Mean                          0.3807   0.1703       0.1612   
          Std                           0.0742   0.0390       0.0365   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.6177                   0.6248   
          Std               0.0657                   0.0609   
Precision Mean              0.6408                   0.6476   
          Std               0.0638                   0.0612   
Recall    Mean              0.6177                   0.6248   
          Std               0.0657                   0.0609   
F1 score  Mean              0.6132                   0.6210   
          Std               0.0697                   0.0649   
ARI       Mean              0.3589                   0.3675   
          Std               0.0818                   0.0737   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.5402  
          Std                     0.0767  
Precision Mean                    0.5140  
          Std                     0.1012  
Recall    Mean                    0.5402  
          Std                     0.0767  
F1 score  Mean                    0.4844  
          Std                     0.0945  
ARI       Mean                    0.3354  
          Std                     0.0771

In [ ]:
# N, V, k, alpha, nmin, max_depth = (2000, 15, 8, 0.5, 50, 4)
tables[6]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.9952  0.8648         0.8790   
          Std                 0.0046  0.0966         0.0984   
Precision Mean                0.9954  0.8208         0.8262   
          Std                 0.0044  0.1329         0.1413   
Recall    Mean                0.9952  0.8648         0.8790   
          Std                 0.0046  0.0966         0.0984   
F1 score  Mean                0.9952  0.8265         0.8417   
          Std                 0.0046  0.1220         0.1275   
ARI       Mean                0.9891  0.8403         0.8594   
          Std                 0.0104  0.1189         0.1166   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.8971   0.6499       0.5176   
          Std                           0.0854   0.1659       0.1415   
Precision Mean                          0.8479   0.5768       0.4959   
          Std                           0.1248   0.1882       0.1404   
Recall    Mean                          0.8971   0.6499       0.5176   
          Std                           0.0854   0.1659       0.1415   
F1 score  Mean                          0.8642   0.5693       0.4304   
          Std                           0.1121   0.1948       0.1522   
ARI       Mean                          0.8893   0.5597       0.3665   
          Std                           0.0916   0.2134       0.1908   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.8052                   0.8575   
          Std               0.0969                   0.0969   
Precision Mean              0.7712                   0.8172   
          Std               0.1238                   0.1313   
Recall    Mean              0.8052                   0.8575   
          Std               0.0969                   0.0969   
F1 score  Mean              0.7567                   0.8178   
          Std               0.1173                   0.1225   
ARI       Mean              0.7545                   0.8309   
          Std               0.1344                   0.1194   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.8512  
          Std                     0.1288  
Precision Mean                    0.7962  
          Std                     0.1716  
Recall    Mean                    0.8512  
          Std                     0.1288  
F1 score  Mean                    0.8104  
          Std                     0.1597  
ARI       Mean                    0.8207  
          Std                     0.1651

In [ ]:
# N, V, k, alpha, nmin, max_depth = (2000, 15, 8, 0.85, 50, 4)
tables[7]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.7804  0.7606         0.6891   
          Std                 0.0346  0.0496         0.0662   
Precision Mean                0.7979  0.7782         0.6715   
          Std                 0.0305  0.0496         0.0896   
Recall    Mean                0.7804  0.7606         0.6891   
          Std                 0.0346  0.0496         0.0662   
F1 score  Mean                0.7807  0.7568         0.6546   
          Std                 0.0346  0.0534         0.0867   
ARI       Mean                0.5760  0.5541         0.5036   
          Std                 0.0577  0.0697         0.0674   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.7479   0.2850       0.3080   
          Std                           0.0533   0.0389       0.0319   
Precision Mean                          0.7652   0.2830       0.3075   
          Std                           0.0524   0.0965       0.0820   
Recall    Mean                          0.7479   0.2850       0.3080   
          Std                           0.0533   0.0389       0.0319   
F1 score  Mean                          0.7374   0.1681       0.2147   
          Std                           0.0624   0.0446       0.0382   
ARI       Mean                          0.5471   0.1616       0.1619   
          Std                           0.0684   0.0365       0.0367   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.7503                   0.7556   
          Std               0.0462                   0.0481   
Precision Mean              0.7706                   0.7716   
          Std               0.0438                   0.0515   
Recall    Mean              0.7503                   0.7556   
          Std               0.0462                   0.0481   
F1 score  Mean              0.7476                   0.7514   
          Std               0.0496                   0.0532   
ARI       Mean              0.5375                   0.5468   
          Std               0.0633                   0.0680   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.5624  
          Std                     0.1007  
Precision Mean                    0.5456  
          Std                     0.1188  
Recall    Mean                    0.5624  
          Std                     0.1007  
F1 score  Mean                    0.4950  
          Std                     0.1257  
ARI       Mean                    0.3855  
          Std                     0.0984

In [ ]:
# N, V, k, alpha, nmin, max_depth = (2000, 6, 15, 0.5, 50, 4)
tables[8]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.8796  0.6746         0.5869   
          Std                 0.0552  0.1011         0.1087   
Precision Mean                0.8468  0.6079         0.4903   
          Std                 0.0770  0.1133         0.1238   
Recall    Mean                0.8796  0.6746         0.5869   
          Std                 0.0552  0.1011         0.1087   
F1 score  Mean                0.8534  0.6027         0.4925   
          Std                 0.0702  0.1158         0.1190   
ARI       Mean                0.8349  0.5870         0.5198   
          Std                 0.0676  0.1436         0.1472   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.6960   0.2942       0.2998   
          Std                           0.1113   0.1054       0.0714   
Precision Mean                          0.6194   0.2450       0.2721   
          Std                           0.1295   0.0937       0.0935   
Recall    Mean                          0.6960   0.2942       0.2998   
          Std                           0.1113   0.1054       0.0714   
F1 score  Mean                          0.6262   0.1999       0.2200   
          Std                           0.1284   0.1040       0.0752   
ARI       Mean                          0.6158   0.1933       0.1543   
          Std                           0.1491   0.1202       0.0833   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.6538                   0.6748   
          Std               0.0992                   0.1004   
Precision Mean              0.5972                   0.6062   
          Std               0.1115                   0.1172   
Recall    Mean              0.6538                   0.6748   
          Std               0.0992                   0.1004   
F1 score  Mean              0.5818                   0.6026   
          Std               0.1117                   0.1148   
ARI       Mean              0.5577                   0.5893   
          Std               0.1406                   0.1440   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.4119  
          Std                     0.1286  
Precision Mean                    0.3528  
          Std                     0.1315  
Recall    Mean                    0.4119  
          Std                     0.1286  
F1 score  Mean                    0.3119  
          Std                     0.1343  
ARI       Mean                    0.3206  
          Std                     0.1527

In [ ]:
# N, V, k, alpha, nmin, max_depth = (2000, 6, 15, 0.85, 50, 4)
tables[9]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.4739  0.4633         0.4049   
          Std                 0.0502  0.0475         0.0458   
Precision Mean                0.4245  0.4322         0.3116   
          Std                 0.0603  0.0570         0.0628   
Recall    Mean                0.4739  0.4633         0.4049   
          Std                 0.0502  0.0475         0.0458   
F1 score  Mean                0.4301  0.4200         0.3224   
          Std                 0.0543  0.0518         0.0506   
ARI       Mean                0.2793  0.2562         0.2485   
          Std                 0.0477  0.0478         0.0440   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.4292   0.1888       0.1998   
          Std                           0.0457   0.0292       0.0231   
Precision Mean                          0.3533   0.1340       0.1348   
          Std                           0.0699   0.0478       0.0349   
Recall    Mean                          0.4292   0.1888       0.1998   
          Std                           0.0457   0.0292       0.0231   
F1 score  Mean                          0.3601   0.1002       0.1209   
          Std                           0.0536   0.0275       0.0202   
ARI       Mean                          0.2540   0.0923       0.0984   
          Std                           0.0431   0.0251       0.0230   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.4562                   0.4620   
          Std               0.0497                   0.0483   
Precision Mean              0.4302                   0.4300   
          Std               0.0552                   0.0572   
Recall    Mean              0.4562                   0.4620   
          Std               0.0497                   0.0483   
F1 score  Mean              0.4169                   0.4191   
          Std               0.0536                   0.0540   
ARI       Mean              0.2495                   0.2543   
          Std               0.0446                   0.0481   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.3079  
          Std                     0.0550  
Precision Mean                    0.2111  
          Std                     0.0795  
Recall    Mean                    0.3079  
          Std                     0.0550  
F1 score  Mean                    0.2039  
          Std                     0.0632  
ARI       Mean                    0.1850  
          Std                     0.0439

In [ ]:
# N, V, k, alpha, nmin, max_depth = (2000, 15, 15, 0.5, 50, 4)
tables[10]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.9474  0.6480         0.6145   
          Std                 0.0353  0.0879         0.1084   
Precision Mean                0.9336  0.5717         0.5038   
          Std                 0.0539  0.0894         0.1270   
Recall    Mean                0.9474  0.6480         0.6145   
          Std                 0.0353  0.0879         0.1084   
F1 score  Mean                0.9353  0.5748         0.5253   
          Std                 0.0473  0.0947         0.1247   
ARI       Mean                0.9292  0.5431         0.5371   
          Std                 0.0370  0.1405         0.1347   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.6591   0.3253       0.2896   
          Std                           0.1006   0.1076       0.0767   
Precision Mean                          0.5505   0.2650       0.2669   
          Std                           0.1176   0.0921       0.0881   
Recall    Mean                          0.6591   0.3253       0.2896   
          Std                           0.1006   0.1076       0.0767   
F1 score  Mean                          0.5769   0.2331       0.2159   
          Std                           0.1165   0.1062       0.0749   
ARI       Mean                          0.5968   0.2061       0.1242   
          Std                           0.1339   0.1185       0.0809   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.6155                   0.6475   
          Std               0.0857                   0.0917   
Precision Mean              0.5543                   0.5635   
          Std               0.0879                   0.0965   
Recall    Mean              0.6155                   0.6475   
          Std               0.0857                   0.0917   
F1 score  Mean              0.5424                   0.5733   
          Std               0.0944                   0.0999   
ARI       Mean              0.4907                   0.5482   
          Std               0.1188                   0.1416   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.4464  
          Std                     0.1454  
Precision Mean                    0.3598  
          Std                     0.1406  
Recall    Mean                    0.4464  
          Std                     0.1454  
F1 score  Mean                    0.3474  
          Std                     0.1545  
ARI       Mean                    0.3574  
          Std                     0.1668

In [ ]:
# N, V, k, alpha, nmin, max_depth = (2000, 15, 15, 0.85, 50, 4)
tables[11]

entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.5874  0.5635         0.4405   
          Std                 0.0363  0.0448         0.0453   
Precision Mean                0.5525  0.5459         0.3252   
          Std                 0.0464  0.0561         0.0658   
Recall    Mean                0.5874  0.5635         0.4405   
          Std                 0.0363  0.0448         0.0453   
F1 score  Mean                0.5515  0.5263         0.3460   
          Std                 0.0389  0.0540         0.0560   
ARI       Mean                0.3844  0.3514         0.2866   
          Std                 0.0441  0.0476         0.0407   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.5032   0.1560       0.2049   
          Std                           0.0556   0.0172       0.0268   
Precision Mean                          0.4515   0.1214       0.1576   
          Std                           0.0826   0.0508       0.0496   
Recall    Mean                          0.5032   0.1560       0.2049   
          Std                           0.0556   0.0172       0.0268   
F1 score  Mean                          0.4406   0.0651       0.1257   
          Std                           0.0696   0.0158       0.0279   
ARI       Mean                          0.3110   0.0777       0.0994   
          Std                           0.0453   0.0138       0.0201   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.5520                   0.5619   
          Std               0.0464                   0.0440   
Precision Mean              0.5347                   0.5425   
          Std               0.0548                   0.0564   
Recall    Mean              0.5520                   0.5619   
          Std               0.0464                   0.0440   
F1 score  Mean              0.5162                   0.5256   
          Std               0.0535                   0.0527   
ARI       Mean              0.3382                   0.3498   
          Std               0.0499                   0.0484   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.2442  
          Std                     0.0465  
Precision Mean                    0.1498  
          Std                     0.0590  
Recall    Mean                    0.2442  
          Std                     0.0465  
F1 score  Mean                    0.1288  
          Std                     0.0448  
ARI       Mean                    0.1510  
          Std                     0.0422

---
## Real World Datasets (UCI Repository)

### Yeast

In [164]:
df = pd.read_csv('../Datasets/yeast.csv')
X = df.drop("target",axis=1).values
le = LabelEncoder()
y = le.fit_transform(df["target"])

In [165]:
compare_metrics_train_test(max_depth=3, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 3)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.5035  0.5598         0.4387   
          Std                 0.0268  0.0204         0.0279   
Precision Mean                0.3637  0.5488         0.2440   
          Std                 0.0440  0.0250         0.0476   
Recall    Mean                0.5035  0.5598         0.4387   
          Std                 0.0268  0.0204         0.0279   
F1 score  Mean                0.4093  0.5344         0.2950   
          Std                 0.0358  0.0232         0.0279   
ARI       Mean                0.2700  0.2148         0.2244   
          Std                 0.0260  0.0214         0.0238   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.4481   0.3273       0.3405   
          Std                           0.0246   0.0248       0.0378   
Precision Mean                          0.2620   0.1372       0.1656   
          Std                           0.0620   0.0462       0.0787   
Recall    Mean                          0.4481   0.3273       0.3405   
          Std                           0.0246   0.0248       0.0378   
F1 score  Mean                          0.3042   0.1712       0.1995   
          Std                           0.0262   0.0214       0.0587   
ARI       Mean                          0.2259   0.0879       0.0985   
          Std                           0.0217   0.0161       0.0177   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.5521                   0.5552   
          Std               0.0204                   0.0201   
Precision Mean              0.5441                   0.5453   
          Std               0.0220                   0.0242   
Recall    Mean              0.5521                   0.5552   
          Std               0.0204                   0.0201   
F1 score  Mean              0.5281                   0.5299   
          Std               0.0221                   0.0226   
ARI       Mean              0.2089                   0.2108   
          Std               0.0201                   0.0207   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.3345  
          Std                     0.0304  
Precision Mean                    0.1600  
          Std                     0.0543  
Recall    Mean                    0.3345  
          Std                     0.0304  
F1 score  Mean                    0.1809  
          Std                     0.0280  
ARI       Mean                    0.0949  
          Std                     0.0240

In [166]:
compare_metrics_train_test(max_depth=4, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 4)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.5691  0.5650         0.4783   
          Std                 0.0207  0.0204         0.0319   
Precision Mean                0.5631  0.5653         0.3570   
          Std                 0.0226  0.0247         0.0772   
Recall    Mean                0.5691  0.5650         0.4783   
          Std                 0.0207  0.0204         0.0319   
F1 score  Mean                0.5520  0.5461         0.3639   
          Std                 0.0231  0.0227         0.0450   
ARI       Mean                0.2445  0.2365         0.2567   
          Std                 0.0224  0.0226         0.0332   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.4720   0.3340       0.3730   
          Std                           0.0292   0.0336       0.0482   
Precision Mean                          0.3619   0.1840       0.2540   
          Std                           0.0709   0.0663       0.0923   
Recall    Mean                          0.4720   0.3340       0.3730   
          Std                           0.0292   0.0336       0.0482   
F1 score  Mean                          0.3485   0.1836       0.2684   
          Std                           0.0410   0.0322       0.0755   
ARI       Mean                          0.2424   0.1017       0.1161   
          Std                           0.0260   0.0302       0.0402   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.5621                   0.5640   
          Std               0.0214                   0.0209   
Precision Mean              0.5635                   0.5656   
          Std               0.0242                   0.0234   
Recall    Mean              0.5621                   0.5640   
          Std               0.0214                   0.0209   
F1 score  Mean              0.5423                   0.5464   
          Std               0.0234                   0.0231   
ARI       Mean              0.2340                   0.2360   
          Std               0.0212                   0.0224   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.3802  
          Std                     0.0464  
Precision Mean                    0.2285  
          Std                     0.0744  
Recall    Mean                    0.3802  
          Std                     0.0464  
F1 score  Mean                    0.2330  
          Std                     0.0479  
ARI       Mean                    0.1437  
          Std                     0.0580

### Glass

In [167]:
df = pd.read_csv('../Datasets/glass.csv')
X = df.drop('Type', axis=1).to_numpy()
y = df['Type'].to_numpy()

In [168]:
compare_metrics_train_test(max_depth=3, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 3)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.6318  0.6533         0.5456   
          Std                 0.0567  0.0512         0.0953   
Precision Mean                0.6351  0.6040         0.4600   
          Std                 0.0684  0.0646         0.1688   
Recall    Mean                0.6318  0.6533         0.5456   
          Std                 0.0567  0.0512         0.0953   
F1 score  Mean                0.6141  0.6134         0.4603   
          Std                 0.0593  0.0517         0.1334   
ARI       Mean                0.2926  0.3154         0.3067   
          Std                 0.0759  0.0749         0.0712   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.5711   0.4363       0.4144   
          Std                           0.1033   0.0819       0.0557   
Precision Mean                          0.5195   0.3107       0.2822   
          Std                           0.1792   0.1412       0.1363   
Recall    Mean                          0.5711   0.4363       0.4144   
          Std                           0.1033   0.0819       0.0557   
F1 score  Mean                          0.5028   0.3177       0.2847   
          Std                           0.1435   0.1002       0.0688   
ARI       Mean                          0.3092   0.2345       0.2257   
          Std                           0.0725   0.0646       0.0562   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.6207                   0.6415   
          Std               0.0561                   0.0655   
Precision Mean              0.6075                   0.6059   
          Std               0.0703                   0.0731   
Recall    Mean              0.6207                   0.6415   
          Std               0.0561                   0.0655   
F1 score  Mean              0.5952                   0.6076   
          Std               0.0573                   0.0644   
ARI       Mean              0.2857                   0.3003   
          Std               0.0747                   0.0849   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.5074  
          Std                     0.0933  
Precision Mean                    0.3872  
          Std                     0.1525  
Recall    Mean                    0.5074  
          Std                     0.0933  
F1 score  Mean                    0.4139  
          Std                     0.1302  
ARI       Mean                    0.2657  
          Std                     0.0560

In [169]:
compare_metrics_train_test(max_depth=4, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 4)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.6637  0.6578         0.5952   
          Std                 0.0585  0.0569         0.1039   
Precision Mean                0.6804  0.6515         0.5613   
          Std                 0.0578  0.0790         0.1648   
Recall    Mean                0.6637  0.6578         0.5952   
          Std                 0.0585  0.0569         0.1039   
F1 score  Mean                0.6547  0.6386         0.5462   
          Std                 0.0608  0.0632         0.1429   
ARI       Mean                0.3185  0.3119         0.2975   
          Std                 0.0782  0.0825         0.0739   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.6393   0.5081       0.4474   
          Std                           0.0763   0.1200       0.0771   
Precision Mean                          0.6286   0.4577       0.3928   
          Std                           0.1033   0.1731       0.1742   
Recall    Mean                          0.6393   0.5081       0.4474   
          Std                           0.0763   0.1200       0.0771   
F1 score  Mean                          0.6138   0.4273       0.3389   
          Std                           0.0926   0.1549       0.1020   
ARI       Mean                          0.3047   0.2673       0.2400   
          Std                           0.0763   0.0798       0.0605   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.6381                   0.6537   
          Std               0.0579                   0.0702   
Precision Mean              0.6511                   0.6596   
          Std               0.0621                   0.0862   
Recall    Mean              0.6381                   0.6537   
          Std               0.0579                   0.0702   
F1 score  Mean              0.6262                   0.6381   
          Std               0.0590                   0.0767   
ARI       Mean              0.2881                   0.3047   
          Std               0.0729                   0.0927   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.5870  
          Std                     0.1098  
Precision Mean                    0.5392  
          Std                     0.1719  
Recall    Mean                    0.5870  
          Std                     0.1098  
F1 score  Mean                    0.5365  
          Std                     0.1518  
ARI       Mean                    0.2888  
          Std                     0.0804

### Breast-cancer-prognostic

In [170]:
df = pd.read_csv('../DATASETS/breast_cancer_prognostic.csv')
df = df.drop('Unnamed: 0', axis=1)
df['Outcome'] = df['Outcome'].map({'R': 1, 'N': 0})
X = df.drop('Outcome', axis = 1).to_numpy()
y = df['Outcome'].to_numpy()

In [171]:
compare_metrics_train_test(max_depth=3, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 3)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.7098  0.6955         0.6959   
          Std                 0.0870  0.0731         0.0728   
Precision Mean                0.6490  0.6558         0.6562   
          Std                 0.0890  0.0874         0.0872   
Recall    Mean                0.7098  0.6955         0.6959   
          Std                 0.0870  0.0731         0.0728   
F1 score  Mean                0.6654  0.6646         0.6650   
          Std                 0.0777  0.0783         0.0783   
ARI       Mean                0.0076  0.0175         0.0180   
          Std                 0.0699  0.0887         0.0909   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.6955   0.6951       0.6959   
          Std                           0.0723   0.0727       0.0730   
Precision Mean                          0.6561   0.6561       0.6564   
          Std                           0.0870   0.0871       0.0874   
Recall    Mean                          0.6955   0.6951       0.6959   
          Std                           0.0723   0.0727       0.0730   
F1 score  Mean                          0.6647   0.6645       0.6651   
          Std                           0.0778   0.0781       0.0786   
ARI       Mean                          0.0174   0.0174       0.0182   
          Std                           0.0888   0.0890       0.0915   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.6963                   0.6959   
          Std               0.0718                   0.0727   
Precision Mean              0.6564                   0.6547   
          Std               0.0868                   0.0874   
Recall    Mean              0.6963                   0.6959   
          Std               0.0718                   0.0727   
F1 score  Mean              0.6652                   0.6646   
          Std               0.0774                   0.0778   
ARI       Mean              0.0178                   0.0176   
          Std               0.0884                   0.0905   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.6959  
          Std                     0.0735  
Precision Mean                    0.6548  
          Std                     0.0877  
Recall    Mean                    0.6959  
          Std                     0.0735  
F1 score  Mean                    0.6646  
          Std                     0.0783  
ARI       Mean                    0.0181  
          Std                     0.0905

In [172]:
compare_metrics_train_test(max_depth=4, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 4)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.6882  0.6629         0.6620   
          Std                 0.0832  0.0639         0.0648   
Precision Mean                0.6598  0.6516         0.6515   
          Std                 0.0859  0.0800         0.0781   
Recall    Mean                0.6882  0.6629         0.6620   
          Std                 0.0832  0.0639         0.0648   
F1 score  Mean                0.6616  0.6499         0.6496   
          Std                 0.0831  0.0696         0.0700   
ARI       Mean                0.0238  0.0079         0.0071   
          Std                 0.0708  0.0650         0.0618   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.6625   0.6629       0.6620   
          Std                           0.0657   0.0631       0.0658   
Precision Mean                          0.6517   0.6522       0.6516   
          Std                           0.0780   0.0783       0.0784   
Recall    Mean                          0.6625   0.6629       0.6620   
          Std                           0.0657   0.0631       0.0658   
F1 score  Mean                          0.6500   0.6504       0.6498   
          Std                           0.0698   0.0686       0.0701   
ARI       Mean                          0.0073   0.0079       0.0073   
          Std                           0.0621   0.0632       0.0626   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.6625                   0.6616   
          Std               0.0655                   0.0657   
Precision Mean              0.6512                   0.6505   
          Std               0.0781                   0.0791   
Recall    Mean              0.6625                   0.6616   
          Std               0.0655                   0.0657   
F1 score  Mean              0.6500                   0.6490   
          Std               0.0698                   0.0703   
ARI       Mean              0.0073                   0.0065   
          Std               0.0619                   0.0624   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.6637  
          Std                     0.0671  
Precision Mean                    0.6516  
          Std                     0.0782  
Recall    Mean                    0.6637  
          Std                     0.0671  
F1 score  Mean                    0.6506  
          Std                     0.0708  
ARI       Mean                    0.0083  
          Std                     0.0635

### Congressional voting records

In [173]:
df = pd.read_csv('../DATASETS/congressional-voting-records.csv')
X = df.drop('Class Name', axis=1).to_numpy()
y = df['Class Name'].to_numpy()

In [174]:
compare_metrics_train_test(max_depth=3, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 3)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.9507  0.9493         0.9493   
          Std                 0.0272  0.0270         0.0270   
Precision Mean                0.9534  0.9524         0.9524   
          Std                 0.0244  0.0241         0.0241   
Recall    Mean                0.9507  0.9493         0.9493   
          Std                 0.0272  0.0270         0.0270   
F1 score  Mean                0.9506  0.9493         0.9493   
          Std                 0.0272  0.0271         0.0271   
ARI       Mean                0.8121  0.8070         0.8070   
          Std                 0.0973  0.0965         0.0965   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.9493   0.9493       0.9493   
          Std                           0.0270   0.0270       0.0270   
Precision Mean                          0.9524   0.9524       0.9524   
          Std                           0.0241   0.0241       0.0241   
Recall    Mean                          0.9493   0.9493       0.9493   
          Std                           0.0270   0.0270       0.0270   
F1 score  Mean                          0.9493   0.9493       0.9493   
          Std                           0.0271   0.0271       0.0271   
ARI       Mean                          0.8070   0.8070       0.8070   
          Std                           0.0965   0.0965       0.0965   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.9486                   0.9493   
          Std               0.0266                   0.0270   
Precision Mean              0.9517                   0.9524   
          Std               0.0237                   0.0241   
Recall    Mean              0.9486                   0.9493   
          Std               0.0266                   0.0270   
F1 score  Mean              0.9486                   0.9493   
          Std               0.0267                   0.0271   
ARI       Mean              0.8044                   0.8070   
          Std               0.0948                   0.0965   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.9493  
          Std                     0.0270  
Precision Mean                    0.9524  
          Std                     0.0241  
Recall    Mean                    0.9493  
          Std                     0.0270  
F1 score  Mean                    0.9493  
          Std                     0.0271  
ARI       Mean                    0.8070  
          Std                     0.0965

In [175]:
compare_metrics_train_test(max_depth=4, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 4)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.9517  0.9483         0.9497   
          Std                 0.0282  0.0272         0.0271   
Precision Mean                0.9544  0.9513         0.9525   
          Std                 0.0244  0.0241         0.0242   
Recall    Mean                0.9517  0.9483         0.9497   
          Std                 0.0282  0.0272         0.0271   
F1 score  Mean                0.9517  0.9483         0.9496   
          Std                 0.0283  0.0272         0.0272   
ARI       Mean                0.8162  0.8033         0.8083   
          Std                 0.1011  0.0969         0.0970   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.9493   0.9483       0.9486   
          Std                           0.0272   0.0276       0.0275   
Precision Mean                          0.9521   0.9513       0.9517   
          Std                           0.0243   0.0246       0.0244   
Recall    Mean                          0.9493   0.9483       0.9486   
          Std                           0.0272   0.0276       0.0275   
F1 score  Mean                          0.9493   0.9483       0.9486   
          Std                           0.0273   0.0276       0.0275   
ARI       Mean                          0.8071   0.8034       0.8046   
          Std                           0.0975   0.0987       0.0982   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.9490                   0.9483   
          Std               0.0280                   0.0278   
Precision Mean              0.9519                   0.9512   
          Std               0.0251                   0.0248   
Recall    Mean              0.9490                   0.9483   
          Std               0.0280                   0.0278   
F1 score  Mean              0.9490                   0.9483   
          Std               0.0281                   0.0279   
ARI       Mean              0.8060                   0.8035   
          Std               0.1002                   0.0994   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.9490  
          Std                     0.0271  
Precision Mean                    0.9520  
          Std                     0.0239  
Recall    Mean                    0.9490  
          Std                     0.0271  
F1 score  Mean                    0.9489  
          Std                     0.0272  
ARI       Mean                    0.8058  
          Std                     0.0974

### Balance-scale

In [176]:
df = pd.read_csv('../Datasets/balance-scale-preprocessed.csv')
X = df.drop('Class', axis=1).to_numpy()
y = df['Class'].to_numpy()

In [177]:
compare_metrics_train_test(max_depth=3, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 3)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.6989  0.7092         0.7085   
          Std                 0.0266  0.0326         0.0287   
Precision Mean                0.6511  0.6612         0.6617   
          Std                 0.0316  0.0370         0.0334   
Recall    Mean                0.6989  0.7092         0.7085   
          Std                 0.0266  0.0326         0.0287   
F1 score  Mean                0.6707  0.6811         0.6803   
          Std                 0.0292  0.0344         0.0300   
ARI       Mean                0.2219  0.2427         0.2406   
          Std                 0.0457  0.0606         0.0534   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.7085   0.7084       0.6912   
          Std                           0.0310   0.0280       0.0444   
Precision Mean                          0.6600   0.6601       0.6498   
          Std                           0.0355   0.0329       0.0451   
Recall    Mean                          0.7085   0.7084       0.6912   
          Std                           0.0310   0.0280       0.0444   
F1 score  Mean                          0.6804   0.6800       0.6611   
          Std                           0.0326   0.0304       0.0471   
ARI       Mean                          0.2414   0.2403       0.2129   
          Std                           0.0568   0.0505       0.0763   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.7057                   0.7062   
          Std               0.0316                   0.0304   
Precision Mean              0.6576                   0.6573   
          Std               0.0347                   0.0338   
Recall    Mean              0.7057                   0.7062   
          Std               0.0316                   0.0304   
F1 score  Mean              0.6774                   0.6781   
          Std               0.0331                   0.0319   
ARI       Mean              0.2366                   0.2374   
          Std               0.0588                   0.0571   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.7062  
          Std                     0.0307  
Precision Mean                    0.6576  
          Std                     0.0344  
Recall    Mean                    0.7062  
          Std                     0.0307  
F1 score  Mean                    0.6782  
          Std                     0.0323  
ARI       Mean                    0.2372  
          Std                     0.0574

In [178]:
compare_metrics_train_test(max_depth=4, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 4)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.7442  0.7825         0.7757   
          Std                 0.0261  0.0320         0.0293   
Precision Mean                0.7067  0.7334         0.7277   
          Std                 0.0326  0.0357         0.0305   
Recall    Mean                0.7442  0.7825         0.7757   
          Std                 0.0261  0.0320         0.0293   
F1 score  Mean                0.7199  0.7551         0.7483   
          Std                 0.0280  0.0332         0.0290   
ARI       Mean                0.3239  0.4173         0.3993   
          Std                 0.0633  0.0757         0.0724   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.7782   0.7526       0.7027   
          Std                           0.0310   0.0323       0.0523   
Precision Mean                          0.7295   0.7105       0.6758   
          Std                           0.0328   0.0334       0.0485   
Recall    Mean                          0.7782   0.7526       0.7027   
          Std                           0.0310   0.0323       0.0523   
F1 score  Mean                          0.7508   0.7256       0.6748   
          Std                           0.0310   0.0324       0.0538   
ARI       Mean                          0.4063   0.3417       0.2400   
          Std                           0.0744   0.0760       0.0961   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.7731                   0.7766   
          Std               0.0293                   0.0310   
Precision Mean              0.7290                   0.7322   
          Std               0.0385                   0.0397   
Recall    Mean              0.7731                   0.7766   
          Std               0.0293                   0.0310   
F1 score  Mean              0.7464                   0.7499   
          Std               0.0312                   0.0326   
ARI       Mean              0.3922                   0.4016   
          Std               0.0690                   0.0731   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.7721  
          Std                     0.0318  
Precision Mean                    0.7251  
          Std                     0.0345  
Recall    Mean                    0.7721  
          Std                     0.0318  
F1 score  Mean                    0.7449  
          Std                     0.0330  
ARI       Mean                    0.3910  
          Std                     0.0758

### Blood-transfusion

In [179]:
df = pd.read_csv('../Datasets/blood-transfusion.csv')
X = df.drop('whether he/she donated blood in March 2007', axis=1).to_numpy()
y = df['whether he/she donated blood in March 2007'].to_numpy()

In [180]:
compare_metrics_train_test(max_depth=3, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 3)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.7744  0.7757         0.7757   
          Std                 0.0319  0.0295         0.0295   
Precision Mean                0.7516  0.7496         0.7496   
          Std                 0.0490  0.0417         0.0417   
Recall    Mean                0.7744  0.7757         0.7757   
          Std                 0.0319  0.0295         0.0295   
F1 score  Mean                0.7512  0.7473         0.7473   
          Std                 0.0442  0.0453         0.0453   
ARI       Mean                0.1925  0.1829         0.1829   
          Std                 0.0836  0.0856         0.0856   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.7757   0.7757       0.7757   
          Std                           0.0295   0.0295       0.0295   
Precision Mean                          0.7496   0.7496       0.7496   
          Std                           0.0417   0.0417       0.0417   
Recall    Mean                          0.7757   0.7757       0.7757   
          Std                           0.0295   0.0295       0.0295   
F1 score  Mean                          0.7473   0.7473       0.7473   
          Std                           0.0453   0.0453       0.0453   
ARI       Mean                          0.1829   0.1829       0.1829   
          Std                           0.0856   0.0856       0.0856   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.7757                   0.7757   
          Std               0.0295                   0.0295   
Precision Mean              0.7496                   0.7496   
          Std               0.0417                   0.0417   
Recall    Mean              0.7757                   0.7757   
          Std               0.0295                   0.0295   
F1 score  Mean              0.7473                   0.7473   
          Std               0.0453                   0.0453   
ARI       Mean              0.1829                   0.1829   
          Std               0.0856                   0.0856   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.7757  
          Std                     0.0295  
Precision Mean                    0.7496  
          Std                     0.0417  
Recall    Mean                    0.7757  
          Std                     0.0295  
F1 score  Mean                    0.7473  
          Std                     0.0453  
ARI       Mean                    0.1829  
          Std                     0.0856

In [181]:
compare_metrics_train_test(max_depth=4, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 4)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.7725  0.7761         0.7761   
          Std                 0.0306  0.0299         0.0299   
Precision Mean                0.7601  0.7635         0.7635   
          Std                 0.0335  0.0353         0.0353   
Recall    Mean                0.7725  0.7761         0.7761   
          Std                 0.0306  0.0299         0.0299   
F1 score  Mean                0.7585  0.7599         0.7599   
          Std                 0.0354  0.0382         0.0382   
ARI       Mean                0.2084  0.2122         0.2122   
          Std                 0.0679  0.0752         0.0752   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.7761   0.7761       0.7761   
          Std                           0.0299   0.0299       0.0299   
Precision Mean                          0.7635   0.7635       0.7635   
          Std                           0.0354   0.0353       0.0353   
Recall    Mean                          0.7761   0.7761       0.7761   
          Std                           0.0299   0.0299       0.0299   
F1 score  Mean                          0.7599   0.7599       0.7599   
          Std                           0.0382   0.0382       0.0382   
ARI       Mean                          0.2122   0.2122       0.2122   
          Std                           0.0751   0.0752       0.0752   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.7761                   0.7761   
          Std               0.0299                   0.0299   
Precision Mean              0.7635                   0.7635   
          Std               0.0353                   0.0353   
Recall    Mean              0.7761                   0.7761   
          Std               0.0299                   0.0299   
F1 score  Mean              0.7599                   0.7599   
          Std               0.0382                   0.0382   
ARI       Mean              0.2122                   0.2122   
          Std               0.0752                   0.0752   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.7761  
          Std                     0.0299  
Precision Mean                    0.7635  
          Std                     0.0354  
Recall    Mean                    0.7761  
          Std                     0.0299  
F1 score  Mean                    0.7599  
          Std                     0.0382  
ARI       Mean                    0.2122  
          Std                     0.0751

### Car-evaluation

In [ ]:
with open('../Datasets/car+evaluation/car.names', 'r') as f:
    column_names = []
    for line in f:
        line = line.strip()
        if ':' in line and not line.startswith(('|', '#')):
            column_names.append(line.split(':')[0].strip())

df = pd.read_csv('../Datasets/car+evaluation/car.data', header=None, names=column_names, delimiter=',')

df = df[['1. Title', '2. Sources', '(a) Creator', '(b) Donors', '(c) Date', '3. Past Usage','M. Bohanec and V. Rajkovic']]
df.columns = ['buying', 'maint', 'doors', 'persons','lug_boot','safety','class']

for column in df.columns:
    df[column], _ = pd.factorize(df[column])
    
X = df.drop('class', axis=1).to_numpy()
y = df['class'].to_numpy()

In [183]:
compare_metrics_train_test(max_depth=3, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 3)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.7887  0.7878         0.7887   
          Std                 0.0190  0.0190         0.0190   
Precision Mean                0.7300  0.7300         0.7300   
          Std                 0.0271  0.0271         0.0271   
Recall    Mean                0.7887  0.7878         0.7887   
          Std                 0.0190  0.0190         0.0190   
F1 score  Mean                0.7554  0.7549         0.7554   
          Std                 0.0172  0.0170         0.0172   
ARI       Mean                0.5088  0.5061         0.5088   
          Std                 0.0527  0.0530         0.0527   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.7887   0.7656       0.7163   
          Std                           0.0190   0.0214       0.0249   
Precision Mean                          0.7300   0.7637       0.6665   
          Std                           0.0271   0.0560       0.0505   
Recall    Mean                          0.7887   0.7656       0.7163   
          Std                           0.0190   0.0214       0.0249   
F1 score  Mean                          0.7554   0.7483       0.6784   
          Std                           0.0172   0.0248       0.0411   
ARI       Mean                          0.5088   0.4003       0.2121   
          Std                           0.0527   0.0514       0.0695   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.7898                   0.7879   
          Std               0.0146                   0.0166   
Precision Mean              0.7448                   0.7302   
          Std               0.0290                   0.0257   
Recall    Mean              0.7898                   0.7879   
          Std               0.0146                   0.0166   
F1 score  Mean              0.7627                   0.7554   
          Std               0.0185                   0.0158   
ARI       Mean              0.5012                   0.5058   
          Std               0.0393                   0.0480   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.7887  
          Std                     0.0190  
Precision Mean                    0.7300  
          Std                     0.0271  
Recall    Mean                    0.7887  
          Std                     0.0190  
F1 score  Mean                    0.7554  
          Std                     0.0172  
ARI       Mean                    0.5088  
          Std                     0.0527

In [184]:
compare_metrics_train_test(max_depth=4, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 4)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.8339  0.8443         0.8443   
          Std                 0.0183  0.0151         0.0151   
Precision Mean                0.8523  0.8451         0.8458   
          Std                 0.0207  0.0223         0.0222   
Recall    Mean                0.8339  0.8443         0.8443   
          Std                 0.0183  0.0151         0.0151   
F1 score  Mean                0.8308  0.8371         0.8376   
          Std                 0.0198  0.0186         0.0181   
ARI       Mean                0.6506  0.6821         0.6836   
          Std                 0.0554  0.0489         0.0464   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.8443   0.8310       0.7467   
          Std                           0.0151   0.0242       0.0219   
Precision Mean                          0.8458   0.7947       0.7268   
          Std                           0.0222   0.0297       0.0305   
Recall    Mean                          0.8443   0.8310       0.7467   
          Std                           0.0151   0.0242       0.0219   
F1 score  Mean                          0.8376   0.8091       0.7300   
          Std                           0.0181   0.0266       0.0254   
ARI       Mean                          0.6836   0.6187       0.3225   
          Std                           0.0464   0.0591       0.0513   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.8366                   0.8441   
          Std               0.0197                   0.0167   
Precision Mean              0.8384                   0.8458   
          Std               0.0203                   0.0203   
Recall    Mean              0.8366                   0.8441   
          Std               0.0197                   0.0167   
F1 score  Mean              0.8256                   0.8369   
          Std               0.0235                   0.0195   
ARI       Mean              0.6364                   0.6762   
          Std               0.0724                   0.0603   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.8443  
          Std                     0.0151  
Precision Mean                    0.8458  
          Std                     0.0222  
Recall    Mean                    0.8443  
          Std                     0.0151  
F1 score  Mean                    0.8376  
          Std                     0.0181  
ARI       Mean                    0.6836  
          Std                     0.0464

### Connectionist-bench-sonar

In [185]:
df = pd.read_csv('../Datasets/connectionist-bench-sonar.csv')
df['Label'] = df['Label'].map({'R':0, 'M':1})
X = df.drop('Label', axis=1).to_numpy()
y = df['Label'].to_numpy()

In [186]:
compare_metrics_train_test(max_depth=3, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 3)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.7238  0.7162         0.7138   
          Std                 0.0660  0.0671         0.0667   
Precision Mean                0.7475  0.7292         0.7271   
          Std                 0.0703  0.0703         0.0690   
Recall    Mean                0.7238  0.7162         0.7138   
          Std                 0.0660  0.0671         0.0667   
F1 score  Mean                0.7192  0.7133         0.7110   
          Std                 0.0676  0.0688         0.0680   
ARI       Mean                0.2019  0.1892         0.1850   
          Std                 0.1162  0.1148         0.1131   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.7165   0.7127       0.7138   
          Std                           0.0667   0.0661       0.0661   
Precision Mean                          0.7300   0.7266       0.7274   
          Std                           0.0697   0.0687       0.0690   
Recall    Mean                          0.7165   0.7127       0.7138   
          Std                           0.0667   0.0661       0.0661   
F1 score  Mean                          0.7136   0.7096       0.7108   
          Std                           0.0685   0.0678       0.0679   
ARI       Mean                          0.1897   0.1826       0.1846   
          Std                           0.1141   0.1127       0.1125   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.7138                   0.7146   
          Std               0.0661                   0.0670   
Precision Mean              0.7274                   0.7282   
          Std               0.0690                   0.0699   
Recall    Mean              0.7138                   0.7146   
          Std               0.0661                   0.0670   
F1 score  Mean              0.7108                   0.7116   
          Std               0.0679                   0.0687   
ARI       Mean              0.1846                   0.1864   
          Std               0.1125                   0.1146   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.7162  
          Std                     0.0670  
Precision Mean                    0.7296  
          Std                     0.0697  
Recall    Mean                    0.7162  
          Std                     0.0670  
F1 score  Mean                    0.7130  
          Std                     0.0688  
ARI       Mean                    0.1891  
          Std                     0.1139

In [187]:
compare_metrics_train_test(max_depth=4, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 4)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.7235  0.7131         0.7138   
          Std                 0.0640  0.0655         0.0653   
Precision Mean                0.7372  0.7229         0.7233   
          Std                 0.0623  0.0655         0.0646   
Recall    Mean                0.7235  0.7131         0.7138   
          Std                 0.0640  0.0655         0.0653   
F1 score  Mean                0.7219  0.7118         0.7125   
          Std                 0.0640  0.0667         0.0667   
ARI       Mean                0.2005  0.1830         0.1842   
          Std                 0.1120  0.1125         0.1153   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.7115   0.7092       0.7123   
          Std                           0.0658   0.0628       0.0641   
Precision Mean                          0.7215   0.7193       0.7219   
          Std                           0.0663   0.0622       0.0639   
Recall    Mean                          0.7115   0.7092       0.7123   
          Std                           0.0658   0.0628       0.0641   
F1 score  Mean                          0.7103   0.7077       0.7109   
          Std                           0.0669   0.0643       0.0656   
ARI       Mean                          0.1804   0.1749       0.1809   
          Std                           0.1142   0.1071       0.1113   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.7104                   0.7119   
          Std               0.0643                   0.0646   
Precision Mean              0.7197                   0.7217   
          Std               0.0644                   0.0646   
Recall    Mean              0.7104                   0.7119   
          Std               0.0643                   0.0646   
F1 score  Mean              0.7089                   0.7105   
          Std               0.0658                   0.0660   
ARI       Mean              0.1776                   0.1805   
          Std               0.1119                   0.1125   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.7115  
          Std                     0.0664  
Precision Mean                    0.7207  
          Std                     0.0661  
Recall    Mean                    0.7115  
          Std                     0.0664  
F1 score  Mean                    0.7102  
          Std                     0.0676  
ARI       Mean                    0.1807  
          Std                     0.1163

### Contraceptive-method-choice

In [ ]:
with open('../Datasets/contraceptive+method+choice/cmc.names', 'r') as f:
    column_names = []
    for line in f:
        line = line.strip()
        if ':' in line and not line.startswith(('|', '#')):
            column_names.append(line.split(':')[0].strip())

df = pd.read_csv('../Datasets/contraceptive+method+choice/cmc.data', header=None, names=column_names,delimiter=',')
df = df[['1. Title', '2. Sources', '(a) Origin', '(b) Creator', '(c) Donor', '(c) Date', '3. Past Usage', '(ftp', '(http', '4. Relevant Information']]
df.columns = ['w_age', 'w_education','h_education', 'N_children','w_religion','w_working','h_occupation','sol_index','media_exposure','class']

X = df.drop('class', axis=1).to_numpy()
y = df['class'].to_numpy()

In [189]:
compare_metrics_train_test(max_depth=3, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 3)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.5018  0.5118         0.5150   
          Std                 0.0291  0.0269         0.0282   
Precision Mean                0.5463  0.5677         0.5709   
          Std                 0.0771  0.0564         0.0578   
Recall    Mean                0.5018  0.5118         0.5150   
          Std                 0.0291  0.0269         0.0282   
F1 score  Mean                0.4857  0.4992         0.5077   
          Std                 0.0427  0.0317         0.0381   
ARI       Mean                0.0606  0.0676         0.0735   
          Std                 0.0246  0.0229         0.0248   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.5131   0.5125       0.4840   
          Std                           0.0279   0.0291       0.0236   
Precision Mean                          0.5754   0.5418       0.4450   
          Std                           0.0524   0.0699       0.0737   
Recall    Mean                          0.5131   0.5125       0.4840   
          Std                           0.0279   0.0291       0.0236   
F1 score  Mean                          0.5052   0.5040       0.4535   
          Std                           0.0348   0.0469       0.0526   
ARI       Mean                          0.0698   0.0776       0.0758   
          Std                           0.0244   0.0223       0.0184   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.5130                   0.5120   
          Std               0.0311                   0.0276   
Precision Mean              0.5160                   0.5469   
          Std               0.0666                   0.0669   
Recall    Mean              0.5130                   0.5120   
          Std               0.0311                   0.0276   
F1 score  Mean              0.4884                   0.4921   
          Std               0.0426                   0.0374   
ARI       Mean              0.0727                   0.0679   
          Std               0.0270                   0.0262   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.5122  
          Std                     0.0267  
Precision Mean                    0.5728  
          Std                     0.0553  
Recall    Mean                    0.5122  
          Std                     0.0267  
F1 score  Mean                    0.5015  
          Std                     0.0321  
ARI       Mean                    0.0677  
          Std                     0.0232

In [190]:
compare_metrics_train_test(max_depth=4, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 4)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.5436  0.5512         0.5499   
          Std                 0.0260  0.0237         0.0221   
Precision Mean                0.5623  0.5648         0.5566   
          Std                 0.0368  0.0284         0.0235   
Recall    Mean                0.5436  0.5512         0.5499   
          Std                 0.0260  0.0237         0.0221   
F1 score  Mean                0.5374  0.5455         0.5457   
          Std                 0.0324  0.0244         0.0240   
ARI       Mean                0.1026  0.1115         0.1137   
          Std                 0.0302  0.0266         0.0261   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.5497   0.5315       0.5041   
          Std                           0.0227   0.0218       0.0277   
Precision Mean                          0.5618   0.5584       0.4960   
          Std                           0.0260   0.0360       0.0336   
Recall    Mean                          0.5497   0.5315       0.5041   
          Std                           0.0227   0.0218       0.0277   
F1 score  Mean                          0.5450   0.5283       0.4876   
          Std                           0.0242   0.0275       0.0368   
ARI       Mean                          0.1111   0.0935       0.0887   
          Std                           0.0260   0.0234       0.0218   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.5382                   0.5458   
          Std               0.0246                   0.0240   
Precision Mean              0.5597                   0.5616   
          Std               0.0296                   0.0293   
Recall    Mean              0.5382                   0.5458   
          Std               0.0246                   0.0240   
F1 score  Mean              0.5313                   0.5394   
          Std               0.0242                   0.0249   
ARI       Mean              0.0965                   0.1067   
          Std               0.0271                   0.0278   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.5507  
          Std                     0.0231  
Precision Mean                    0.5627  
          Std                     0.0251  
Recall    Mean                    0.5507  
          Std                     0.0231  
F1 score  Mean                    0.5463  
          Std                     0.0237  
ARI       Mean                    0.1119  
          Std                     0.0256

### Haberman-survival

In [191]:
with open('../Datasets/haberman+s+survival/haberman.names', 'r') as f:
    column_names = []
    for line in f:
        line = line.strip()
        if ':' in line and not line.startswith(('|', '#')):
            column_names.append(line.split(':')[0].strip())

df = pd.read_csv('../Datasets/haberman+s+survival/haberman.data', header=None, names=column_names,delimiter=',')
df = df[['1. Title', '2. Sources', '(a) Donor', '(b) Date']]
df.columns = ['age', 'operation_year','positive_auxillary_nodes','survival_status']

X = df.drop('survival_status', axis=1).to_numpy()
y = df['survival_status'].to_numpy()

In [192]:
compare_metrics_train_test(max_depth=3, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 3)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.7343  0.7343         0.7343   
          Std                 0.0465  0.0462         0.0462   
Precision Mean                0.7172  0.7150         0.7150   
          Std                 0.0695  0.0697         0.0697   
Recall    Mean                0.7343  0.7343         0.7343   
          Std                 0.0465  0.0462         0.0462   
F1 score  Mean                0.7107  0.7095         0.7095   
          Std                 0.0631  0.0600         0.0600   
ARI       Mean                0.1289  0.1244         0.1244   
          Std                 0.0914  0.0856         0.0856   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.7343   0.7345       0.7343   
          Std                           0.0462   0.0464       0.0462   
Precision Mean                          0.7150   0.7153       0.7150   
          Std                           0.0697   0.0698       0.0697   
Recall    Mean                          0.7343   0.7345       0.7343   
          Std                           0.0462   0.0464       0.0462   
F1 score  Mean                          0.7095   0.7097       0.7095   
          Std                           0.0600   0.0601       0.0600   
ARI       Mean                          0.1244   0.1248       0.1244   
          Std                           0.0856   0.0855       0.0856   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.7343                   0.7343   
          Std               0.0462                   0.0462   
Precision Mean              0.7150                   0.7150   
          Std               0.0697                   0.0697   
Recall    Mean              0.7343                   0.7343   
          Std               0.0462                   0.0462   
F1 score  Mean              0.7095                   0.7095   
          Std               0.0600                   0.0600   
ARI       Mean              0.1244                   0.1244   
          Std               0.0856                   0.0856   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.7343  
          Std                     0.0462  
Precision Mean                    0.7150  
          Std                     0.0697  
Recall    Mean                    0.7343  
          Std                     0.0462  
F1 score  Mean                    0.7095  
          Std                     0.0600  
ARI       Mean                    0.1244  
          Std                     0.0856

In [193]:
compare_metrics_train_test(max_depth=4, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 4)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.7265  0.7322         0.7327   
          Std                 0.0436  0.0472         0.0470   
Precision Mean                0.7065  0.7239         0.7243   
          Std                 0.0587  0.0556         0.0554   
Recall    Mean                0.7265  0.7322         0.7327   
          Std                 0.0436  0.0472         0.0470   
F1 score  Mean                0.7078  0.7178         0.7183   
          Std                 0.0521  0.0493         0.0491   
ARI       Mean                0.1165  0.1375         0.1383   
          Std                 0.0841  0.0776         0.0770   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.7306   0.7319       0.7319   
          Std                           0.0467   0.0472       0.0473   
Precision Mean                          0.7216   0.7235       0.7234   
          Std                           0.0561   0.0554       0.0557   
Recall    Mean                          0.7306   0.7319       0.7319   
          Std                           0.0467   0.0472       0.0473   
F1 score  Mean                          0.7156   0.7174       0.7175   
          Std                           0.0497   0.0491       0.0493   
ARI       Mean                          0.1334   0.1366       0.1367   
          Std                           0.0766   0.0776       0.0783   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.7325                   0.7325   
          Std               0.0470                   0.0470   
Precision Mean              0.7240                   0.7240   
          Std               0.0555                   0.0555   
Recall    Mean              0.7325                   0.7325   
          Std               0.0470                   0.0470   
F1 score  Mean              0.7180                   0.7180   
          Std               0.0492                   0.0492   
ARI       Mean              0.1379                   0.1379   
          Std               0.0783                   0.0783   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.7327  
          Std                     0.0471  
Precision Mean                    0.7242  
          Std                     0.0554  
Recall    Mean                    0.7327  
          Std                     0.0471  
F1 score  Mean                    0.7182  
          Std                     0.0492  
ARI       Mean                    0.1383  
          Std                     0.0780

### Hayes-roth

In [194]:
df = pd.read_csv('../Datasets/hayes+roth/hayes-roth.data', header=None)
df.columns = ['name','hobby','age','educational level','marital status','class']

X = df.drop(['class','name'],axis=1).to_numpy()
y = df['class'].to_numpy()

In [195]:
compare_metrics_train_test(max_depth=3, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 3)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.5528  0.5503         0.5528   
          Std                 0.0602  0.0628         0.0602   
Precision Mean                0.3704  0.3730         0.3704   
          Std                 0.0585  0.0605         0.0585   
Recall    Mean                0.5528  0.5503         0.5528   
          Std                 0.0602  0.0628         0.0602   
F1 score  Mean                0.4241  0.4243         0.4241   
          Std                 0.0630  0.0630         0.0630   
ARI       Mean                0.4657  0.4584         0.4657   
          Std                 0.0698  0.0893         0.0698   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.5528   0.5528       0.5528   
          Std                           0.0602   0.0602       0.0602   
Precision Mean                          0.3704   0.3704       0.3704   
          Std                           0.0585   0.0585       0.0585   
Recall    Mean                          0.5528   0.5528       0.5528   
          Std                           0.0602   0.0602       0.0602   
F1 score  Mean                          0.4241   0.4241       0.4241   
          Std                           0.0630   0.0630       0.0630   
ARI       Mean                          0.4657   0.4657       0.4657   
          Std                           0.0698   0.0698       0.0698   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.5376                   0.5443   
          Std               0.0517                   0.0549   
Precision Mean              0.5587                   0.3946   
          Std               0.1277                   0.0893   
Recall    Mean              0.5376                   0.5443   
          Std               0.0517                   0.0549   
F1 score  Mean              0.5179                   0.4316   
          Std               0.0894                   0.0607   
ARI       Mean              0.1861                   0.4163   
          Std               0.1467                   0.1445   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.5528  
          Std                     0.0602  
Precision Mean                    0.3704  
          Std                     0.0585  
Recall    Mean                    0.5528  
          Std                     0.0602  
F1 score  Mean                    0.4241  
          Std                     0.0630  
ARI       Mean                    0.4657  
          Std                     0.0698

In [196]:
compare_metrics_train_test(max_depth=4, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 4)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.6564  0.6564         0.6564   
          Std                 0.0448  0.0448         0.0448   
Precision Mean                0.6759  0.6759         0.6759   
          Std                 0.0625  0.0625         0.0625   
Recall    Mean                0.6564  0.6564         0.6564   
          Std                 0.0448  0.0448         0.0448   
F1 score  Mean                0.6551  0.6551         0.6551   
          Std                 0.0476  0.0476         0.0476   
ARI       Mean                0.3449  0.3449         0.3449   
          Std                 0.0757  0.0757         0.0757   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.6564   0.6564       0.6564   
          Std                           0.0448   0.0448       0.0448   
Precision Mean                          0.6759   0.6759       0.6759   
          Std                           0.0625   0.0625       0.0625   
Recall    Mean                          0.6564   0.6564       0.6564   
          Std                           0.0448   0.0448       0.0448   
F1 score  Mean                          0.6551   0.6551       0.6551   
          Std                           0.0476   0.0476       0.0476   
ARI       Mean                          0.3449   0.3449       0.3449   
          Std                           0.0757   0.0757       0.0757   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.6115                   0.6455   
          Std               0.0909                   0.0636   
Precision Mean              0.6764                   0.6696   
          Std               0.0814                   0.0663   
Recall    Mean              0.6115                   0.6455   
          Std               0.0909                   0.0636   
F1 score  Mean              0.6059                   0.6443   
          Std               0.0943                   0.0651   
ARI       Mean              0.2542                   0.3300   
          Std               0.1144                   0.0874   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.6564  
          Std                     0.0448  
Precision Mean                    0.6759  
          Std                     0.0625  
Recall    Mean                    0.6564  
          Std                     0.0448  
F1 score  Mean                    0.6551  
          Std                     0.0476  
ARI       Mean                    0.3449  
          Std                     0.0757

### Heart-disease-Cleveland

In [197]:
df = pd.read_csv('../Datasets/Heart_disease_cleveland_new.csv')
X = df.drop('target', axis=1).to_numpy()
y = df['target'].to_numpy()

In [198]:
compare_metrics_train_test(max_depth=3, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 3)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.7821  0.7858         0.7858   
          Std                 0.0454  0.0412         0.0412   
Precision Mean                0.7885  0.7904         0.7904   
          Std                 0.0436  0.0429         0.0429   
Recall    Mean                0.7821  0.7858         0.7858   
          Std                 0.0454  0.0412         0.0412   
F1 score  Mean                0.7804  0.7846         0.7846   
          Std                 0.0463  0.0412         0.0412   
ARI       Mean                0.3175  0.3244         0.3244   
          Std                 0.1003  0.0955         0.0955   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.7858   0.7858       0.7858   
          Std                           0.0412   0.0412       0.0412   
Precision Mean                          0.7904   0.7904       0.7904   
          Std                           0.0429   0.0429       0.0429   
Recall    Mean                          0.7858   0.7858       0.7858   
          Std                           0.0412   0.0412       0.0412   
F1 score  Mean                          0.7846   0.7846       0.7846   
          Std                           0.0412   0.0412       0.0412   
ARI       Mean                          0.3244   0.3244       0.3244   
          Std                           0.0955   0.0955       0.0955   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.7858                   0.7858   
          Std               0.0412                   0.0412   
Precision Mean              0.7904                   0.7904   
          Std               0.0429                   0.0429   
Recall    Mean              0.7858                   0.7858   
          Std               0.0412                   0.0412   
F1 score  Mean              0.7846                   0.7846   
          Std               0.0412                   0.0412   
ARI       Mean              0.3244                   0.3244   
          Std               0.0955                   0.0955   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.7858  
          Std                     0.0412  
Precision Mean                    0.7904  
          Std                     0.0429  
Recall    Mean                    0.7858  
          Std                     0.0412  
F1 score  Mean                    0.7846  
          Std                     0.0412  
ARI       Mean                    0.3244  
          Std                     0.0955

In [199]:
compare_metrics_train_test(max_depth=4, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 4)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.7613  0.7466         0.7479   
          Std                 0.0449  0.0427         0.0447   
Precision Mean                0.7720  0.7565         0.7577   
          Std                 0.0450  0.0433         0.0451   
Recall    Mean                0.7613  0.7466         0.7479   
          Std                 0.0449  0.0427         0.0447   
F1 score  Mean                0.7585  0.7444         0.7457   
          Std                 0.0455  0.0433         0.0453   
ARI       Mean                0.2715  0.2404         0.2438   
          Std                 0.0950  0.0823         0.0869   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.7482   0.7482       0.7482   
          Std                           0.0447   0.0442       0.0442   
Precision Mean                          0.7581   0.7580       0.7580   
          Std                           0.0451   0.0445       0.0445   
Recall    Mean                          0.7482   0.7482       0.7482   
          Std                           0.0447   0.0442       0.0442   
F1 score  Mean                          0.7460   0.7460       0.7460   
          Std                           0.0453   0.0449       0.0449   
ARI       Mean                          0.2443   0.2441       0.2441   
          Std                           0.0868   0.0863       0.0863   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.7484                   0.7482   
          Std               0.0446                   0.0446   
Precision Mean              0.7582                   0.7579   
          Std               0.0449                   0.0449   
Recall    Mean              0.7484                   0.7482   
          Std               0.0446                   0.0446   
F1 score  Mean              0.7462                   0.7460   
          Std               0.0453                   0.0452   
ARI       Mean              0.2448                   0.2443   
          Std               0.0876                   0.0875   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.7479  
          Std                     0.0438  
Precision Mean                    0.7576  
          Std                     0.0441  
Recall    Mean                    0.7479  
          Std                     0.0438  
F1 score  Mean                    0.7457  
          Std                     0.0445  
ARI       Mean                    0.2435  
          Std                     0.0853

### Hepatitis

In [200]:
df = pd.read_csv('../Datasets/hepatitis_csv.csv')
df = df.dropna()
df = df.replace({True: 1, False: 0})
df['sex'] = df['sex'].map({'female': 0, 'male': 1})
df['class'] = df['class'].map({'live': 1, 'die': 0})

X = df.drop('class',axis=1).to_numpy()
y = df['class'].to_numpy()

In [201]:
compare_metrics_train_test(max_depth=3, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 3)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.8170  0.8110         0.8080   
          Std                 0.0704  0.0666         0.0681   
Precision Mean                0.8340  0.8166         0.8137   
          Std                 0.0949  0.0977         0.1001   
Recall    Mean                0.8170  0.8110         0.8080   
          Std                 0.0704  0.0666         0.0681   
F1 score  Mean                0.8122  0.8071         0.8038   
          Std                 0.0810  0.0783         0.0805   
ARI       Mean                0.2194  0.1992         0.1937   
          Std                 0.2126  0.1894         0.1905   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.8080   0.8080       0.8060   
          Std                           0.0681   0.0681       0.0668   
Precision Mean                          0.8137   0.8134       0.8136   
          Std                           0.1001   0.1000       0.1000   
Recall    Mean                          0.8080   0.8080       0.8060   
          Std                           0.0681   0.0681       0.0668   
F1 score  Mean                          0.8038   0.8036       0.8027   
          Std                           0.0805   0.0806       0.0796   
ARI       Mean                          0.1937   0.1915       0.1934   
          Std                           0.1905   0.1890       0.1910   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.8120                   0.8080   
          Std               0.0690                   0.0681   
Precision Mean              0.8198                   0.8137   
          Std               0.0994                   0.1001   
Recall    Mean              0.8120                   0.8080   
          Std               0.0690                   0.0681   
F1 score  Mean              0.8087                   0.8038   
          Std               0.0795                   0.0805   
ARI       Mean              0.2084                   0.1937   
          Std               0.1978                   0.1905   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.8090  
          Std                     0.0691  
Precision Mean                    0.8152  
          Std                     0.0994  
Recall    Mean                    0.8090  
          Std                     0.0691  
F1 score  Mean                    0.8053  
          Std                     0.0804  
ARI       Mean                    0.1986  
          Std                     0.1973

In [202]:
compare_metrics_train_test(max_depth=4, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 4)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.8180  0.8100         0.8050   
          Std                 0.0705  0.0686         0.0709   
Precision Mean                0.8373  0.8280         0.8214   
          Std                 0.0962  0.0919         0.0979   
Recall    Mean                0.8180  0.8100         0.8050   
          Std                 0.0705  0.0686         0.0709   
F1 score  Mean                0.8141  0.8129         0.8068   
          Std                 0.0827  0.0763         0.0804   
ARI       Mean                0.2288  0.2294         0.2151   
          Std                 0.2173  0.1911         0.1956   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.8060   0.8070       0.8080   
          Std                           0.0712   0.0700       0.0724   
Precision Mean                          0.8233   0.8248       0.8252   
          Std                           0.0961   0.0947       0.0948   
Recall    Mean                          0.8060   0.8070       0.8080   
          Std                           0.0712   0.0700       0.0724   
F1 score  Mean                          0.8080   0.8093       0.8100   
          Std                           0.0797   0.0789       0.0796   
ARI       Mean                          0.2178   0.2217       0.2238   
          Std                           0.1987   0.1921       0.1921   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.8100                   0.8070   
          Std               0.0721                   0.0707   
Precision Mean              0.8307                   0.8251   
          Std               0.0935                   0.0948   
Recall    Mean              0.8100                   0.8070   
          Std               0.0721                   0.0707   
F1 score  Mean              0.8136                   0.8094   
          Std               0.0778                   0.0789   
ARI       Mean              0.2356                   0.2228   
          Std               0.1989                   0.1935   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.8040  
          Std                     0.0720  
Precision Mean                    0.8224  
          Std                     0.0972  
Recall    Mean                    0.8040  
          Std                     0.0720  
F1 score  Mean                    0.8069  
          Std                     0.0803  
ARI       Mean                    0.2153  
          Std                     0.1950

### Ionosphere

In [203]:
df = pd.read_csv('../Datasets/ionosphere/ionosphere.data', header=None,delimiter=',')
df = pd.get_dummies(df, drop_first=True, dtype=int)
X = df.drop('34_g', axis=1).to_numpy()
y = df['34_g'].to_numpy()

In [204]:
compare_metrics_train_test(max_depth=3, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 3)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.8984  0.8866         0.8868   
          Std                 0.0372  0.0418         0.0415   
Precision Mean                0.9028  0.8924         0.8927   
          Std                 0.0312  0.0368         0.0362   
Recall    Mean                0.8984  0.8866         0.8868   
          Std                 0.0372  0.0418         0.0415   
F1 score  Mean                0.8978  0.8857         0.8859   
          Std                 0.0395  0.0445         0.0442   
ARI       Mean                0.6325  0.5963         0.5970   
          Std                 0.1144  0.1265         0.1255   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.8868   0.8866       0.8866   
          Std                           0.0415   0.0418       0.0418   
Precision Mean                          0.8927   0.8924       0.8924   
          Std                           0.0362   0.0368       0.0368   
Recall    Mean                          0.8868   0.8866       0.8866   
          Std                           0.0415   0.0418       0.0418   
F1 score  Mean                          0.8859   0.8857       0.8857   
          Std                           0.0442   0.0445       0.0445   
ARI       Mean                          0.5970   0.5963       0.5963   
          Std                           0.1255   0.1265       0.1265   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.8866                   0.8866   
          Std               0.0418                   0.0418   
Precision Mean              0.8924                   0.8924   
          Std               0.0368                   0.0368   
Recall    Mean              0.8866                   0.8866   
          Std               0.0418                   0.0418   
F1 score  Mean              0.8857                   0.8857   
          Std               0.0445                   0.0445   
ARI       Mean              0.5963                   0.5963   
          Std               0.1265                   0.1265   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.8866  
          Std                     0.0416  
Precision Mean                    0.8924  
          Std                     0.0368  
Recall    Mean                    0.8866  
          Std                     0.0416  
F1 score  Mean                    0.8857  
          Std                     0.0444  
ARI       Mean                    0.5963  
          Std                     0.1258

In [205]:
compare_metrics_train_test(max_depth=4, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 4)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.8761  0.8820         0.8825   
          Std                 0.0362  0.0322         0.0306   
Precision Mean                0.8809  0.8860         0.8864   
          Std                 0.0366  0.0323         0.0306   
Recall    Mean                0.8761  0.8820         0.8825   
          Std                 0.0362  0.0322         0.0306   
F1 score  Mean                0.8731  0.8795         0.8801   
          Std                 0.0373  0.0335         0.0318   
ARI       Mean                0.5581  0.5757         0.5769   
          Std                 0.1108  0.1016         0.0975   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.8820   0.8809       0.8818   
          Std                           0.0318   0.0327       0.0318   
Precision Mean                          0.8862   0.8850       0.8857   
          Std                           0.0316   0.0325       0.0318   
Recall    Mean                          0.8820   0.8809       0.8818   
          Std                           0.0318   0.0327       0.0318   
F1 score  Mean                          0.8795   0.8784       0.8793   
          Std                           0.0330   0.0338       0.0331   
ARI       Mean                          0.5756   0.5724       0.5749   
          Std                           0.0998   0.1028       0.1005   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.8820                   0.8816   
          Std               0.0322                   0.0322   
Precision Mean              0.8860                   0.8854   
          Std               0.0323                   0.0323   
Recall    Mean              0.8820                   0.8816   
          Std               0.0322                   0.0322   
F1 score  Mean              0.8795                   0.8791   
          Std               0.0335                   0.0334   
ARI       Mean              0.5757                   0.5743   
          Std               0.1016                   0.1016   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.8845  
          Std                     0.0321  
Precision Mean                    0.8886  
          Std                     0.0318  
Recall    Mean                    0.8845  
          Std                     0.0321  
F1 score  Mean                    0.8820  
          Std                     0.0332  
ARI       Mean                    0.5836  
          Std                     0.1021

### Mammographic Mass

In [223]:
df = pd.read_csv('../Datasets/Mammographic Mass.csv')
X = df.drop('Severity', axis=1).to_numpy()
y = df['Severity'].to_numpy()

In [224]:
compare_metrics_train_test(max_depth=3, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 3)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.8393  0.8430         0.8428   
          Std                 0.0213  0.0228         0.0226   
Precision Mean                0.8448  0.8485         0.8484   
          Std                 0.0201  0.0214         0.0212   
Recall    Mean                0.8393  0.8430         0.8428   
          Std                 0.0213  0.0228         0.0226   
F1 score  Mean                0.8385  0.8421         0.8419   
          Std                 0.0217  0.0231         0.0229   
ARI       Mean                0.4598  0.4700         0.4695   
          Std                 0.0571  0.0618         0.0613   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.8428   0.8428       0.8428   
          Std                           0.0226   0.0226       0.0226   
Precision Mean                          0.8484   0.8484       0.8484   
          Std                           0.0212   0.0212       0.0212   
Recall    Mean                          0.8428   0.8428       0.8428   
          Std                           0.0226   0.0226       0.0226   
F1 score  Mean                          0.8419   0.8419       0.8419   
          Std                           0.0229   0.0229       0.0229   
ARI       Mean                          0.4695   0.4695       0.4695   
          Std                           0.0613   0.0613       0.0613   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.8426                   0.8428   
          Std               0.0226                   0.0228   
Precision Mean              0.8482                   0.8483   
          Std               0.0213                   0.0214   
Recall    Mean              0.8426                   0.8428   
          Std               0.0226                   0.0228   
F1 score  Mean              0.8417                   0.8419   
          Std               0.0229                   0.0231   
ARI       Mean              0.4689                   0.4695   
          Std               0.0613                   0.0618   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.8426  
          Std                     0.0226  
Precision Mean                    0.8482  
          Std                     0.0213  
Recall    Mean                    0.8426  
          Std                     0.0226  
F1 score  Mean                    0.8417  
          Std                     0.0229  
ARI       Mean                    0.4689  
          Std                     0.0613

In [208]:
compare_metrics_train_test(max_depth=4, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 4)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.8308  0.8262         0.8263   
          Std                 0.0245  0.0232         0.0230   
Precision Mean                0.8356  0.8314         0.8316   
          Std                 0.0239  0.0228         0.0227   
Recall    Mean                0.8308  0.8262         0.8263   
          Std                 0.0245  0.0232         0.0230   
F1 score  Mean                0.8301  0.8257         0.8257   
          Std                 0.0246  0.0232         0.0231   
ARI       Mean                0.4373  0.4251         0.4254   
          Std                 0.0638  0.0596         0.0593   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.8262   0.8262       0.8262   
          Std                           0.0231   0.0232       0.0232   
Precision Mean                          0.8314   0.8313       0.8313   
          Std                           0.0228   0.0228       0.0228   
Recall    Mean                          0.8262   0.8262       0.8262   
          Std                           0.0231   0.0232       0.0232   
F1 score  Mean                          0.8256   0.8256       0.8256   
          Std                           0.0231   0.0232       0.0232   
ARI       Mean                          0.4249   0.4249       0.4249   
          Std                           0.0594   0.0596       0.0596   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.8262                   0.8261   
          Std               0.0232                   0.0231   
Precision Mean              0.8313                   0.8312   
          Std               0.0229                   0.0228   
Recall    Mean              0.8262                   0.8261   
          Std               0.0232                   0.0231   
F1 score  Mean              0.8256                   0.8255   
          Std               0.0233                   0.0232   
ARI       Mean              0.4249                   0.4246   
          Std               0.0597                   0.0594   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.8261  
          Std                     0.0231  
Precision Mean                    0.8312  
          Std                     0.0228  
Recall    Mean                    0.8261  
          Std                     0.0231  
F1 score  Mean                    0.8255  
          Std                     0.0232  
ARI       Mean                    0.4246  
          Std                     0.0594

### Monks

In [209]:
df = pd.read_csv('../Datasets/monk.csv')
df = df.drop('id', axis=1)
X = df.drop("'class'", axis=1).to_numpy()
y = df["'class'"].to_numpy()

In [210]:
compare_metrics_train_test(max_depth=3, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 3)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.6156  0.6119         0.6119   
          Std                 0.0359  0.0352         0.0352   
Precision Mean                0.5433  0.5455         0.5455   
          Std                 0.0840  0.0815         0.0815   
Recall    Mean                0.6156  0.6119         0.6119   
          Std                 0.0359  0.0352         0.0352   
F1 score  Mean                0.5568  0.5585         0.5585   
          Std                 0.0515  0.0511         0.0511   
ARI       Mean                0.0117  0.0114         0.0114   
          Std                 0.0225  0.0227         0.0227   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.6119   0.6119       0.6119   
          Std                           0.0352   0.0352       0.0352   
Precision Mean                          0.5455   0.5455       0.5455   
          Std                           0.0815   0.0815       0.0815   
Recall    Mean                          0.6119   0.6119       0.6119   
          Std                           0.0352   0.0352       0.0352   
F1 score  Mean                          0.5585   0.5585       0.5585   
          Std                           0.0511   0.0511       0.0511   
ARI       Mean                          0.0114   0.0114       0.0114   
          Std                           0.0227   0.0227       0.0227   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.6119                   0.6119   
          Std               0.0352                   0.0352   
Precision Mean              0.5455                   0.5455   
          Std               0.0815                   0.0815   
Recall    Mean              0.6119                   0.6119   
          Std               0.0352                   0.0352   
F1 score  Mean              0.5585                   0.5585   
          Std               0.0511                   0.0511   
ARI       Mean              0.0114                   0.0114   
          Std               0.0227                   0.0227   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.6119  
          Std                     0.0352  
Precision Mean                    0.5455  
          Std                     0.0815  
Recall    Mean                    0.6119  
          Std                     0.0352  
F1 score  Mean                    0.5585  
          Std                     0.0511  
ARI       Mean                    0.0114  
          Std                     0.0227

In [211]:
compare_metrics_train_test(max_depth=4, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 4)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.6126  0.6151         0.6151   
          Std                 0.0351  0.0376         0.0376   
Precision Mean                0.6047  0.6137         0.6137   
          Std                 0.0485  0.0468         0.0468   
Recall    Mean                0.6126  0.6151         0.6151   
          Std                 0.0351  0.0376         0.0376   
F1 score  Mean                0.5946  0.6022         0.6022   
          Std                 0.0450  0.0449         0.0449   
ARI       Mean                0.0321  0.0387         0.0387   
          Std                 0.0309  0.0312         0.0312   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.6151   0.6146       0.6151   
          Std                           0.0376   0.0369       0.0376   
Precision Mean                          0.6137   0.6133       0.6137   
          Std                           0.0468   0.0462       0.0468   
Recall    Mean                          0.6151   0.6146       0.6151   
          Std                           0.0376   0.0369       0.0376   
F1 score  Mean                          0.6022   0.6017       0.6022   
          Std                           0.0449   0.0442       0.0449   
ARI       Mean                          0.0387   0.0380       0.0387   
          Std                           0.0312   0.0302       0.0312   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.6151                   0.6151   
          Std               0.0376                   0.0376   
Precision Mean              0.6137                   0.6137   
          Std               0.0468                   0.0468   
Recall    Mean              0.6151                   0.6151   
          Std               0.0376                   0.0376   
F1 score  Mean              0.6022                   0.6022   
          Std               0.0449                   0.0449   
ARI       Mean              0.0387                   0.0387   
          Std               0.0312                   0.0312   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.6146  
          Std                     0.0369  
Precision Mean                    0.6133  
          Std                     0.0462  
Recall    Mean                    0.6146  
          Std                     0.0369  
F1 score  Mean                    0.6017  
          Std                     0.0442  
ARI       Mean                    0.0380  
          Std                     0.0302

### Spambase

In [225]:
df = pd.read_csv('../Datasets/spambase.csv')
X = df.drop('spam', axis=1).to_numpy()
y = df['spam'].to_numpy()

In [226]:
compare_metrics_train_test(max_depth=3, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 3)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.8693  0.8820         0.8820   
          Std                 0.0086  0.0111         0.0111   
Precision Mean                0.8757  0.8832         0.8832   
          Std                 0.0085  0.0105         0.0105   
Recall    Mean                0.8693  0.8820         0.8820   
          Std                 0.0086  0.0111         0.0111   
F1 score  Mean                0.8666  0.8810         0.8810   
          Std                 0.0094  0.0113         0.0113   
ARI       Mean                0.5429  0.5824         0.5824   
          Std                 0.0258  0.0337         0.0337   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.8820   0.8820       0.8820   
          Std                           0.0111   0.0111       0.0111   
Precision Mean                          0.8832   0.8832       0.8832   
          Std                           0.0105   0.0105       0.0105   
Recall    Mean                          0.8820   0.8820       0.8820   
          Std                           0.0111   0.0111       0.0111   
F1 score  Mean                          0.8810   0.8810       0.8810   
          Std                           0.0113   0.0113       0.0113   
ARI       Mean                          0.5824   0.5824       0.5824   
          Std                           0.0337   0.0337       0.0337   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.8820                   0.8820   
          Std               0.0111                   0.0111   
Precision Mean              0.8832                   0.8832   
          Std               0.0105                   0.0105   
Recall    Mean              0.8820                   0.8820   
          Std               0.0111                   0.0111   
F1 score  Mean              0.8810                   0.8810   
          Std               0.0113                   0.0113   
ARI       Mean              0.5824                   0.5824   
          Std               0.0337                   0.0337   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.8820  
          Std                     0.0111  
Precision Mean                    0.8832  
          Std                     0.0105  
Recall    Mean                    0.8820  
          Std                     0.0111  
F1 score  Mean                    0.8810  
          Std                     0.0113  
ARI       Mean                    0.5824  
          Std                     0.0337

In [227]:
compare_metrics_train_test(max_depth=4, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 4)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.8982  0.8976         0.8976   
          Std                 0.0086  0.0104         0.0104   
Precision Mean                0.8997  0.8995         0.8995   
          Std                 0.0084  0.0105         0.0105   
Recall    Mean                0.8982  0.8976         0.8976   
          Std                 0.0086  0.0104         0.0104   
F1 score  Mean                0.8969  0.8964         0.8964   
          Std                 0.0087  0.0104         0.0104   
ARI       Mean                0.6326  0.6310         0.6310   
          Std                 0.0271  0.0330         0.0330   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.8976   0.8976       0.8976   
          Std                           0.0104   0.0104       0.0104   
Precision Mean                          0.8995   0.8995       0.8995   
          Std                           0.0105   0.0105       0.0105   
Recall    Mean                          0.8976   0.8976       0.8976   
          Std                           0.0104   0.0104       0.0104   
F1 score  Mean                          0.8964   0.8964       0.8964   
          Std                           0.0104   0.0104       0.0104   
ARI       Mean                          0.6309   0.6310       0.6310   
          Std                           0.0330   0.0330       0.0330   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.8976                   0.8976   
          Std               0.0104                   0.0104   
Precision Mean              0.8995                   0.8995   
          Std               0.0105                   0.0105   
Recall    Mean              0.8976                   0.8976   
          Std               0.0104                   0.0104   
F1 score  Mean              0.8964                   0.8964   
          Std               0.0104                   0.0104   
ARI       Mean              0.6310                   0.6310   
          Std               0.0330                   0.0330   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.8976  
          Std                     0.0104  
Precision Mean                    0.8995  
          Std                     0.0105  
Recall    Mean                    0.8976  
          Std                     0.0104  
F1 score  Mean                    0.8964  
          Std                     0.0104  
ARI       Mean                    0.6310  
          Std                     0.0330

### Bach Choral Harmony

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

df = pd.read_csv('/Users/user/HSE/Articles/1_Family_of_Classifiying_criteria/Datasets/bach_choral_harmony.csv')

X = df.drop(columns=["target"]).to_numpy()
y = df["target"].to_numpy()
encoder_X = OrdinalEncoder()
X = encoder_X.fit_transform(X)

encoder_y = LabelEncoder()
y = encoder_y.fit_transform(y)

In [220]:
compare_metrics_train_test(max_depth=3, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 3)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.4086  0.4162         0.2412   
          Std                 0.0233  0.0115         0.0108   
Precision Mean                0.2463  0.2659         0.0844   
          Std                 0.0211  0.0132         0.0109   
Recall    Mean                0.4086  0.4162         0.2412   
          Std                 0.0233  0.0115         0.0108   
F1 score  Mean                0.2930  0.3098         0.1146   
          Std                 0.0236  0.0122         0.0088   
ARI       Mean                0.2690  0.2675         0.1145   
          Std                 0.0246  0.0163         0.0122   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.2711   0.1524       0.2876   
          Std                           0.0364   0.0226       0.0300   
Precision Mean                          0.1152   0.0405       0.1191   
          Std                           0.0272   0.0118       0.0262   
Recall    Mean                          0.2711   0.1524       0.2876   
          Std                           0.0364   0.0226       0.0300   
F1 score  Mean                          0.1486   0.0573       0.1618   
          Std                           0.0361   0.0150       0.0309   
ARI       Mean                          0.1327   0.0416       0.1521   
          Std                           0.0306   0.0169       0.0173   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.4146                   0.4054   
          Std               0.0140                   0.0139   
Precision Mean              0.2693                   0.2569   
          Std               0.0153                   0.0168   
Recall    Mean              0.4146                   0.4054   
          Std               0.0140                   0.0139   
F1 score  Mean              0.3119                   0.2990   
          Std               0.0145                   0.0164   
ARI       Mean              0.2564                   0.2467   
          Std               0.0155                   0.0240   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.1692  
          Std                     0.0219  
Precision Mean                    0.0522  
          Std                     0.0185  
Recall    Mean                    0.1692  
          Std                     0.0219  
F1 score  Mean                    0.0700  
          Std                     0.0162  
ARI       Mean                    0.0525  
          Std                     0.0128

In [221]:
compare_metrics_train_test(max_depth=4, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 4)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.5416  0.5422         0.3168   
          Std                 0.0186  0.0121         0.0205   
Precision Mean                0.4174  0.4388         0.1566   
          Std                 0.0261  0.0186         0.0295   
Recall    Mean                0.5416  0.5422         0.3168   
          Std                 0.0186  0.0121         0.0205   
F1 score  Mean                0.4586  0.4687         0.1891   
          Std                 0.0239  0.0142         0.0224   
ARI       Mean                0.3979  0.4051         0.1747   
          Std                 0.0252  0.0181         0.0189   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.3752   0.1960       0.3434   
          Std                           0.0574   0.0273       0.0363   
Precision Mean                          0.2226   0.0653       0.1798   
          Std                           0.0624   0.0188       0.0345   
Recall    Mean                          0.3752   0.1960       0.3434   
          Std                           0.0574   0.0273       0.0363   
F1 score  Mean                          0.2608   0.0865       0.2214   
          Std                           0.0643   0.0207       0.0390   
ARI       Mean                          0.2183   0.0796       0.2131   
          Std                           0.0566   0.0235       0.0194   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.5281                   0.5260   
          Std               0.0186                   0.0183   
Precision Mean              0.4285                   0.4159   
          Std               0.0290                   0.0278   
Recall    Mean              0.5281                   0.5260   
          Std               0.0186                   0.0183   
F1 score  Mean              0.4566                   0.4474   
          Std               0.0225                   0.0232   
ARI       Mean              0.3790                   0.3794   
          Std               0.0319                   0.0313   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.2114  
          Std                     0.0152  
Precision Mean                    0.0774  
          Std                     0.0179  
Recall    Mean                    0.2114  
          Std                     0.0152  
F1 score  Mean                    0.1008  
          Std                     0.0128  
ARI       Mean                    0.0918  
          Std                     0.0138

### Led Display Domain

In [ ]:
SEGMENTS = np.array([
    [1,1,1,0,1,1,1],  # 0
    [0,0,1,0,0,1,0],  # 1
    [1,0,1,1,1,0,1],  # 2
    [1,0,1,1,0,1,1],  # 3
    [0,1,1,1,0,1,0],  # 4
    [1,1,0,1,0,1,1],  # 5
    [1,1,0,1,1,1,1],  # 6
    [1,0,1,0,0,1,0],  # 7
    [1,1,1,1,1,1,1],  # 8
    [1,1,1,1,0,1,1],  # 9
])

def generate_led(n_samples=1000, noise=0.1, seed=42):
    rng = np.random.default_rng(seed)
    X = []
    y = []
    for _ in range(n_samples):
        digit = rng.integers(0, 10)
        sample = SEGMENTS[digit].copy()
        for i in range(7):
            if rng.random() < noise:
                sample[i] = 1 - sample[i]
        X.append(sample)
        y.append(digit)

    return np.array(X), np.array(y)


X, y = generate_led(n_samples=1000,noise=0.1,seed=42)
df = pd.DataFrame(X, columns=[f"x{i+1}" for i in range(7)])
df["label"] = y
df.to_csv('led_dispaly_domain.csv',index=False)

X = df.drop('label', axis=1).to_numpy()
y = df['label'].to_numpy()

In [232]:
compare_metrics_train_test(max_depth=3, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 3)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.5616  0.5324         0.5487   
          Std                 0.0399  0.0357         0.0483   
Precision Mean                0.4861  0.4543         0.4718   
          Std                 0.0489  0.0518         0.0640   
Recall    Mean                0.5616  0.5324         0.5487   
          Std                 0.0399  0.0357         0.0483   
F1 score  Mean                0.5082  0.4749         0.4936   
          Std                 0.0465  0.0453         0.0578   
ARI       Mean                0.3782  0.3494         0.3605   
          Std                 0.0403  0.0353         0.0480   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.5478   0.3038       0.3110   
          Std                           0.0468   0.0425       0.0357   
Precision Mean                          0.4717   0.1990       0.2066   
          Std                           0.0611   0.0438       0.0481   
Recall    Mean                          0.5478   0.3038       0.3110   
          Std                           0.0468   0.0425       0.0357   
F1 score  Mean                          0.4928   0.2120       0.2134   
          Std                           0.0556   0.0410       0.0336   
ARI       Mean                          0.3609   0.1854       0.2054   
          Std                           0.0471   0.0454       0.0431   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.5148                   0.5337   
          Std               0.0467                   0.0342   
Precision Mean              0.4324                   0.4528   
          Std               0.0603                   0.0505   
Recall    Mean              0.5148                   0.5337   
          Std               0.0467                   0.0342   
F1 score  Mean              0.4514                   0.4740   
          Std               0.0549                   0.0436   
ARI       Mean              0.3461                   0.3538   
          Std               0.0460                   0.0370   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.5395  
          Std                     0.0525  
Precision Mean                    0.4632  
          Std                     0.0619  
Recall    Mean                    0.5395  
          Std                     0.0525  
F1 score  Mean                    0.4839  
          Std                     0.0597  
ARI       Mean                    0.3602  
          Std                     0.0458

In [216]:
compare_metrics_train_test(max_depth=4, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 4)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.6999  0.6735         0.6717   
          Std                 0.0290  0.0404         0.0438   
Precision Mean                0.7247  0.6937         0.6908   
          Std                 0.0361  0.0510         0.0546   
Recall    Mean                0.6999  0.6735         0.6717   
          Std                 0.0290  0.0404         0.0438   
F1 score  Mean                0.7022  0.6708         0.6682   
          Std                 0.0323  0.0473         0.0518   
ARI       Mean                0.4485  0.4299         0.4304   
          Std                 0.0379  0.0430         0.0411   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.6711   0.3931       0.4246   
          Std                           0.0452   0.0324       0.0476   
Precision Mean                          0.6920   0.3448       0.3895   
          Std                           0.0551   0.0498       0.0661   
Recall    Mean                          0.6711   0.3931       0.4246   
          Std                           0.0452   0.0324       0.0476   
F1 score  Mean                          0.6680   0.3207       0.3611   
          Std                           0.0519   0.0360       0.0543   
ARI       Mean                          0.4287   0.2644       0.2809   
          Std                           0.0454   0.0321       0.0307   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.6677                   0.6732   
          Std               0.0533                   0.0432   
Precision Mean              0.6857                   0.6907   
          Std               0.0687                   0.0594   
Recall    Mean              0.6677                   0.6732   
          Std               0.0533                   0.0432   
F1 score  Mean              0.6626                   0.6697   
          Std               0.0662                   0.0524   
ARI       Mean              0.4269                   0.4276   
          Std               0.0500                   0.0462   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.6691  
          Std                     0.0458  
Precision Mean                    0.6875  
          Std                     0.0599  
Recall    Mean                    0.6691  
          Std                     0.0458  
F1 score  Mean                    0.6653  
          Std                     0.0550  
ARI       Mean                    0.4281  
          Std                     0.0398

### Mushroom

In [236]:
columns = ['class','cap-shape', 'cap-surface', 'cap-color', 'bruises?', 'odor','gill-attachment',
           'gill-spacing', 'gill-size', 'gill-color','stalk-shape', 'stalk-root', 'stalk-surface-above-ring',
           'stalk-surface-below-ring', 'stalk-color-above-ring','stalk-color-below-ring', 'veil-type', 'veil-color',
           'ring-number', 'ring-type', 'spore-print-color','population', 'habitat']

df = pd.read_csv('../Datasets/mushroom/agaricus-lepiota.data',header=None,names=columns)
X_df = df.drop(columns='class').copy()

for column in X_df.columns:
    X_df[column] = LabelEncoder().fit_transform(X_df[column])

X = X_df.to_numpy()
y = LabelEncoder().fit_transform(df['class'])

In [237]:
compare_metrics_train_test(max_depth=3, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 3)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.9568  0.9588         0.9588   
          Std                 0.0036  0.0041         0.0041   
Precision Mean                0.9574  0.9592         0.9592   
          Std                 0.0035  0.0040         0.0040   
Recall    Mean                0.9568  0.9588         0.9588   
          Std                 0.0036  0.0041         0.0041   
F1 score  Mean                0.9568  0.9588         0.9588   
          Std                 0.0036  0.0041         0.0041   
ARI       Mean                0.8346  0.8419         0.8419   
          Std                 0.0133  0.0149         0.0149   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.9588   0.9588       0.9588   
          Std                           0.0041   0.0041       0.0041   
Precision Mean                          0.9592   0.9592       0.9592   
          Std                           0.0040   0.0040       0.0040   
Recall    Mean                          0.9588   0.9588       0.9588   
          Std                           0.0041   0.0041       0.0041   
F1 score  Mean                          0.9588   0.9588       0.9588   
          Std                           0.0041   0.0041       0.0041   
ARI       Mean                          0.8419   0.8419       0.8419   
          Std                           0.0149   0.0149       0.0149   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.9588                   0.9588   
          Std               0.0041                   0.0041   
Precision Mean              0.9592                   0.9592   
          Std               0.0040                   0.0040   
Recall    Mean              0.9588                   0.9588   
          Std               0.0041                   0.0041   
F1 score  Mean              0.9588                   0.9588   
          Std               0.0041                   0.0041   
ARI       Mean              0.8419                   0.8419   
          Std               0.0149                   0.0149   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.9588  
          Std                     0.0041  
Precision Mean                    0.9592  
          Std                     0.0040  
Recall    Mean                    0.9588  
          Std                     0.0041  
F1 score  Mean                    0.9588  
          Std                     0.0041  
ARI       Mean                    0.8419  
          Std                     0.0149

In [238]:
compare_metrics_train_test(max_depth=4, X=X, y=y)


N, V, k, alpha, nmin, max_depth = (None, None, None, None, None, 4)


entropy_sklearn   b = 1  b = p_l ^ 0.5  \
Metric    Statistic                                           
Accuracy  Mean                0.9582  0.9774         0.9774   
          Std                 0.0059  0.0038         0.0038   
Precision Mean                0.9601  0.9777         0.9777   
          Std                 0.0053  0.0035         0.0035   
Recall    Mean                0.9582  0.9774         0.9774   
          Std                 0.0059  0.0038         0.0038   
F1 score  Mean                0.9582  0.9774         0.9774   
          Std                 0.0059  0.0038         0.0038   
ARI       Mean                0.8399  0.9117         0.9117   
          Std                 0.0216  0.0142         0.0142   

                     b = (p_l*(1 - p_l)) ^ 0.5  b = p_l  b = p_l ^ 2  \
Metric    Statistic                                                    
Accuracy  Mean                          0.9774   0.9774       0.9774   
          Std                           0.0038   0.0038       0.0038   
Precision Mean                          0.9777   0.9777       0.9777   
          Std                           0.0035   0.0035       0.0035   
Recall    Mean                          0.9774   0.9774       0.9774   
          Std                           0.0038   0.0038       0.0038   
F1 score  Mean                          0.9774   0.9774       0.9774   
          Std                           0.0038   0.0038       0.0038   
ARI       Mean                          0.9117   0.9117       0.9117   
          Std                           0.0142   0.0142       0.0142   

                     b = -log(p_l)  b = -p_l^0.5 * log(p_l)  \
Metric    Statistic                                           
Accuracy  Mean              0.9774                   0.9774   
          Std               0.0038                   0.0038   
Precision Mean              0.9777                   0.9777   
          Std               0.0035                   0.0035   
Recall    Mean              0.9774                   0.9774   
          Std               0.0038                   0.0038   
F1 score  Mean              0.9774                   0.9774   
          Std               0.0038                   0.0038   
ARI       Mean              0.9117                   0.9117   
          Std               0.0142                   0.0142   

                     b = -p_l * log(p_l)  
Metric    Statistic                       
Accuracy  Mean                    0.9774  
          Std                     0.0038  
Precision Mean                    0.9777  
          Std                     0.0035  
Recall    Mean                    0.9774  
          Std                     0.0038  
F1 score  Mean                    0.9774  
          Std                     0.0038  
ARI       Mean                    0.9117  
          Std                     0.0142